In [ ]:
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
import seaborn as sns
import textwrap
import warnings
import os
warnings.filterwarnings('ignore')

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

#Load the data

In [ ]:
!git clone https://github.com/edzq/SciER.git

In [ ]:

import os

for root, dirs, files in os.walk("/content/SciER"):
    level = root.replace("/content/SciER", "").count(os.sep)
    indent = " " * 2 * level
    print(indent + os.path.basename(root) + "/")
    for file in files[:10]:
        print(indent + "  " + file)

In [ ]:
!find /content/SciER -type f | head -100

In [ ]:
!find /content/SciER -type f \( -name "*.json" -o -name "*.jsonl" -o -name "*.txt" \)

In [ ]:
train_path = "/content/SciER/SciER/LLM/train.jsonl"
dev_path = "/content/SciER/SciER/LLM/dev.jsonl"
test_path = "/content/SciER/SciER/LLM/test.jsonl"
test_ood_path = "/content/SciER/SciER/LLM/test_ood.jsonl"

In [ ]:
import json
import pandas as pd

def load_jsonl(path):
    records = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                records.append(json.loads(line))
    return records

train_records = load_jsonl(train_path)
dev_records = load_jsonl(dev_path)
test_records = load_jsonl(test_path)
test_ood_records = load_jsonl(test_ood_path)

print("Train:", len(train_records))
print("Dev:", len(dev_records))
print("Test:", len(test_records))
print("Test OOD:", len(test_ood_records))

In [ ]:
print(json.dumps(train_records[0], indent=2, ensure_ascii=False)[:4000])

In [ ]:
train_records[0].keys()

In [ ]:
train_df_raw = pd.DataFrame(train_records)
dev_df_raw = pd.DataFrame(dev_records)
test_df_raw = pd.DataFrame(test_records)
test_ood_df_raw = pd.DataFrame(test_ood_records)

train_df_raw.head()

In [ ]:
def convert_ner_to_entities(ner_list):
    entities = []

    if not isinstance(ner_list, list):
        return entities

    for item in ner_list:
        if isinstance(item, list) and len(item) >= 2:
            entities.append({
                "text": item[0],
                "type": item[1]
            })

    return entities


def convert_rel_to_relations(rel_list):
    relations = []

    if not isinstance(rel_list, list):
        return relations

    for item in rel_list:
        if isinstance(item, list) and len(item) >= 3:
            relations.append({
                "head": item[0],
                "relation": item[1],
                "tail": item[2]
            })

    return relations


def build_scier_text_to_json_example(record):
    sentence = record.get("sentence", "")

    entities = convert_ner_to_entities(record.get("ner", []))
    relations = convert_rel_to_relations(record.get("rel", []))

    target = {
        "entities": entities,
        "relations": relations
    }

    return {
        "doc_id": record.get("doc_id", ""),
        "input_text": (
            "Extract scientific entities and relations as JSON from the sentence: "
            + sentence
        ),
        "sentence": sentence,
        "target": target,
        "target_json": json.dumps(target, ensure_ascii=False, sort_keys=True),
        "num_entities": len(entities),
        "num_relations": len(relations),
        "entity_types": [e["type"] for e in entities],
        "relation_types": [r["relation"] for r in relations]
    }

In [ ]:
train_examples = [build_scier_text_to_json_example(r) for r in train_records]
dev_examples = [build_scier_text_to_json_example(r) for r in dev_records]
test_examples = [build_scier_text_to_json_example(r) for r in test_records]
test_ood_examples = [build_scier_text_to_json_example(r) for r in test_ood_records]

train_df = pd.DataFrame(train_examples)
dev_df = pd.DataFrame(dev_examples)
test_df = pd.DataFrame(test_examples)
test_ood_df = pd.DataFrame(test_ood_examples)

train_df.head()

In [ ]:
for i in range(4):
    print("INPUT:")
    print(train_df.iloc[i]["input_text"])
    print("\nTARGET JSON:")
    print(train_df.iloc[i]["target_json"])
    print("=" * 100)

In [ ]:
split_sizes = pd.DataFrame({
    "split": ["train", "dev", "test", "test_ood"],
    "count": [len(train_df), len(dev_df), len(test_df), len(test_ood_df)]
})

split_sizes

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(7, 4))
plt.bar(split_sizes["split"], split_sizes["count"])
plt.xlabel("Split")
plt.ylabel("Number of examples")
plt.title("SciER Dataset Split Sizes", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
train_df["num_entities"].hist(bins=20)
plt.xlabel("Number of entities")
plt.ylabel("Number of examples")
plt.title("Distribution of Entities per Sentence", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
train_df["num_relations"].hist(bins=20)
plt.xlabel("Number of relations")
plt.ylabel("Number of examples")
plt.title("Distribution of Relations per Sentence", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
train_df["num_relations"].hist(bins=20)
plt.xlabel("Number of relations")
plt.ylabel("Number of examples")
plt.title("Distribution of Relations per Sentence", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
from collections import Counter

entity_counter = Counter()

for types in train_df["entity_types"]:
    entity_counter.update(types)

entity_type_df = pd.DataFrame(
    entity_counter.most_common(),
    columns=["entity_type", "count"]
)

entity_type_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(entity_type_df["entity_type"][::-1], entity_type_df["count"][::-1])
plt.xlabel("Count")
plt.ylabel("Entity type")
plt.title("Entity Type Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
relation_counter = Counter()

for types in train_df["relation_types"]:
    relation_counter.update(types)

relation_type_df = pd.DataFrame(
    relation_counter.most_common(),
    columns=["relation_type", "count"]
)

relation_type_df

In [ ]:
plt.figure(figsize=(8, 5))
plt.barh(relation_type_df["relation_type"][::-1], relation_type_df["count"][::-1])
plt.xlabel("Count")
plt.ylabel("Relation type")
plt.title("Relation Type Distribution", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
import os

output_dir = "/content/drive/MyDrive/NLP_26_Project/SciER_processed"
os.makedirs(output_dir, exist_ok=True)

train_df.to_json(f"{output_dir}/train_text_to_json.jsonl", orient="records", lines=True, force_ascii=False)
dev_df.to_json(f"{output_dir}/dev_text_to_json.jsonl", orient="records", lines=True, force_ascii=False)
test_df.to_json(f"{output_dir}/test_text_to_json.jsonl", orient="records", lines=True, force_ascii=False)
test_ood_df.to_json(f"{output_dir}/test_ood_text_to_json.jsonl", orient="records", lines=True, force_ascii=False)

print("Saved to:", output_dir)

In [ ]:
from collections import Counter

entity_type_counter = Counter()

for types in train_df["entity_types"]:
    entity_type_counter.update(types)

entity_type_counter

Before training our models, first we want to be sure our metrics are working and the dataset format in json is corrrect.

##  Evaluation Metrics

The task is evaluated using structured extraction metrics:

- **Valid JSON rate:** percentage of outputs that can be parsed as JSON.
- **Exact match rate:** percentage of examples where entities and relations both exactly match the gold annotation.
- **Entity micro/macro-style example F1:** evaluates extracted entity spans and types.
- **Relation micro/example F1:** evaluates extracted relation triples.
- **Final equal-weighted F1:** average of entity F1 and relation F1.
- **Relation-weighted final F1:** gives more weight to relation extraction.
- **Partial entity recall:** gives credit for partially correct entity spans.

In [ ]:
import json
import re
import numpy as np
import pandas as pd


def parse_json_safe(text):

    if isinstance(text, dict):
        return text, True

    try:
        obj = json.loads(text)
        if isinstance(obj, dict):
            obj.setdefault("entities", [])
            obj.setdefault("relations", [])
            return obj, True
        return {"entities": [], "relations": []}, False

    except Exception:
        return {"entities": [], "relations": []}, False


def normalize_text(text):

    text = str(text).lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def entity_set(obj):

    entities = obj.get("entities", [])
    output = set()

    if not isinstance(entities, list):
        return output

    for ent in entities:
        if isinstance(ent, dict):
            text = normalize_text(ent.get("text", ""))
            ent_type = normalize_text(ent.get("type", ""))

            if text and ent_type:
                output.add((text, ent_type))

    return output


def relation_set(obj):

    relations = obj.get("relations", [])
    output = set()

    if not isinstance(relations, list):
        return output

    for rel in relations:
        if isinstance(rel, dict):
            head = normalize_text(rel.get("head", ""))
            relation = normalize_text(rel.get("relation", ""))
            tail = normalize_text(rel.get("tail", ""))

            if head and relation and tail:
                output.add((head, relation, tail))

    return output


def prf_from_counts(correct, predicted, gold):

    if gold == 0 and predicted == 0:
        return 1.0, 1.0, 1.0

    precision = correct / predicted if predicted > 0 else 0.0
    recall = correct / gold if gold > 0 else 0.0

    if precision + recall == 0:
        f1 = 0.0
    else:
        f1 = 2 * precision * recall / (precision + recall)

    return precision, recall, f1

In [ ]:
def evaluate_structured_predictions(
    df,
    gold_col="target_json",
    pred_col="prediction_text"
):
    valid_json_scores = []
    exact_match_scores = []

    # Micro counts for entities
    entity_correct_total = 0
    entity_pred_total = 0
    entity_gold_total = 0

    # Micro counts for relations
    relation_correct_total = 0
    relation_pred_total = 0
    relation_gold_total = 0

    # Example-level scores
    entity_example_precision = []
    entity_example_recall = []
    entity_example_f1 = []

    relation_example_precision = []
    relation_example_recall = []
    relation_example_f1 = []

    for _, row in df.iterrows():
        gold_obj, _ = parse_json_safe(row[gold_col])
        pred_obj, is_valid = parse_json_safe(row[pred_col])

        valid_json_scores.append(1 if is_valid else 0)

        gold_entities = entity_set(gold_obj)
        pred_entities = entity_set(pred_obj)

        gold_relations = relation_set(gold_obj)
        pred_relations = relation_set(pred_obj)

        # Exact match means all entities and relations are exactly correct
        exact_match = (
            gold_entities == pred_entities and
            gold_relations == pred_relations
        )
        exact_match_scores.append(1 if exact_match else 0)

        # Entity micro counts
        entity_correct = len(gold_entities & pred_entities)
        entity_correct_total += entity_correct
        entity_pred_total += len(pred_entities)
        entity_gold_total += len(gold_entities)

        # Relation micro counts
        relation_correct = len(gold_relations & pred_relations)
        relation_correct_total += relation_correct
        relation_pred_total += len(pred_relations)
        relation_gold_total += len(gold_relations)

        # Entity example-level score
        ep, er, ef1 = prf_from_counts(
            correct=entity_correct,
            predicted=len(pred_entities),
            gold=len(gold_entities)
        )
        entity_example_precision.append(ep)
        entity_example_recall.append(er)
        entity_example_f1.append(ef1)

        # Relation example-level score
        rp, rr, rf1 = prf_from_counts(
            correct=relation_correct,
            predicted=len(pred_relations),
            gold=len(gold_relations)
        )
        relation_example_precision.append(rp)
        relation_example_recall.append(rr)
        relation_example_f1.append(rf1)

    # Final micro scores
    entity_micro_p, entity_micro_r, entity_micro_f1 = prf_from_counts(
        correct=entity_correct_total,
        predicted=entity_pred_total,
        gold=entity_gold_total
    )

    relation_micro_p, relation_micro_r, relation_micro_f1 = prf_from_counts(
        correct=relation_correct_total,
        predicted=relation_pred_total,
        gold=relation_gold_total
    )

    return {
        "valid_json_rate": float(np.mean(valid_json_scores)),
        "exact_match_rate": float(np.mean(exact_match_scores)),

        "entity_micro_precision": entity_micro_p,
        "entity_micro_recall": entity_micro_r,
        "entity_micro_f1": entity_micro_f1,

        "entity_example_precision": float(np.mean(entity_example_precision)),
        "entity_example_recall": float(np.mean(entity_example_recall)),
        "entity_example_f1": float(np.mean(entity_example_f1)),

        "relation_micro_precision": relation_micro_p,
        "relation_micro_recall": relation_micro_r,
        "relation_micro_f1": relation_micro_f1,

        "relation_example_precision": float(np.mean(relation_example_precision)),
        "relation_example_recall": float(np.mean(relation_example_recall)),
        "relation_example_f1": float(np.mean(relation_example_f1)),
    }

The empty baseline obtains non-zero relation example-level F1 because some sentences contain no annotated relations, and predicting an empty relation list is correct for those cases. However, relation micro-F1 remains zero because the baseline does not recover any actual relation instances.

#1. Equal-weighted final score

To obtain a single final score, I combine entity and relation extraction performance. Since the task requires both detecting scientific entities and extracting relations between them, I report a final F1 score as the average of entity F1 and relation F1. I compute this both at the micro level and at the example level.

In [ ]:
def add_equal_weighted_final_scores(metrics):

    metrics = dict(metrics)

    metrics["final_micro_f1_equal_weighted"] = (
        0.5 * metrics["entity_micro_f1"] +
        0.5 * metrics["relation_micro_f1"]
    )

    metrics["final_example_f1_equal_weighted"] = (
        0.5 * metrics["entity_example_f1"] +
        0.5 * metrics["relation_example_f1"]
    )

    return metrics

#2. Relation-weighted final score

Final Score = 0.4 × Entity F1 + 0.6 × Relation F1

This gives more importance to relations because relation extraction is usually harder.

In [ ]:
def add_relation_weighted_final_scores(metrics):

    metrics = dict(metrics)

    metrics["final_micro_f1_relation_weighted"] = (
        0.4 * metrics["entity_micro_f1"] +
        0.6 * metrics["relation_micro_f1"]
    )

    metrics["final_example_f1_relation_weighted"] = (
        0.4 * metrics["entity_example_f1"] +
        0.6 * metrics["relation_example_f1"]
    )

    return metrics

# 3.partial entity recall

This tells you how many gold entities were at least partially found.

In [ ]:
def partial_entity_recall(gold_entities, pred_entities):
    if len(gold_entities) == 0:
        return 1.0 if len(pred_entities) == 0 else 0.0

    matched = 0

    for gold_ent in gold_entities:
        for pred_ent in pred_entities:
            if partial_entity_match(gold_ent, pred_ent):
                matched += 1
                break

    return matched / len(gold_entities)

In [ ]:
def compute_partial_entity_recall_for_df(df, gold_col="target_json", pred_col="prediction_text"):
    recalls = []

    for _, row in df.iterrows():
        gold_obj, _ = parse_json_safe(row[gold_col])
        pred_obj, _ = parse_json_safe(row[pred_col])

        gold_entities = entity_set(gold_obj)
        pred_entities = entity_set(pred_obj)

        recall = partial_entity_recall(gold_entities, pred_entities)
        recalls.append(recall)

    return sum(recalls) / len(recalls)

In [ ]:
def add_partial_entity_recall(metrics, df):
    metrics = dict(metrics)
    metrics["partial_entity_recall"] = compute_partial_entity_recall_for_df(df)
    return metrics

In [ ]:
def partial_entity_recall(gold_entities, pred_entities):
    if len(gold_entities) == 0:
        return 1.0 if len(pred_entities) == 0 else 0.0

    matched = 0

    for gold_ent in gold_entities:
        for pred_ent in pred_entities:
            if partial_entity_match(gold_ent, pred_ent):
                matched += 1
                break

    return matched / len(gold_entities)

In [ ]:
def compute_partial_entity_recall_for_df(df, gold_col="target_json", pred_col="prediction_text"):
    recalls = []

    for _, row in df.iterrows():
        gold_obj, _ = parse_json_safe(row[gold_col])
        pred_obj, _ = parse_json_safe(row[pred_col])

        gold_entities = entity_set(gold_obj)
        pred_entities = entity_set(pred_obj)

        recall = partial_entity_recall(gold_entities, pred_entities)
        recalls.append(recall)

    return sum(recalls) / len(recalls)

# Baselines

This section includes simple non-LLM baselines:
- Empty baseline
- Rule-based baseline

These baselines provide lower-bound references for entity and relation extraction.

##1.Empty baseline

In [ ]:
empty_df = test_df.copy()

empty_df["prediction_text"] = json.dumps({
    "entities": [],
    "relations": []
})

empty_metrics = evaluate_structured_predictions(empty_df)

empty_metrics = add_equal_weighted_final_scores(empty_metrics)
empty_metrics = add_relation_weighted_final_scores(empty_metrics)
empty_metrics = add_partial_entity_recall(empty_metrics, empty_df)

empty_metrics

In [ ]:
baseline_results = pd.DataFrame([
    {"model": "Empty baseline", "split": "test", **empty_metrics},
    {"model": "Empty baseline", "split": "test_ood", **empty_ood_metrics}
])

baseline_results

##2.Rule-based baseline

In [ ]:
KNOWN_TASK_KEYWORDS = [
    "classification",
    "detection",
    "object detection",
    "segmentation",
    "translation",
    "summarization",
    "question answering",
    "named entity recognition",
    "relation extraction",
    "image classification",
    "machine translation",
    "speech recognition"
]

KNOWN_DATASETS = [
    "ImageNet",
    "COCO",
    "SQuAD",
    "MNIST",
    "CIFAR",
    "GLUE",
    "MultiNLI",
    "WikiText"
]


def rule_based_extract(sentence):
    entities = []
    relations = []

    sentence_str = str(sentence)

    for dataset in KNOWN_DATASETS:
        if re.search(rf"\b{re.escape(dataset)}\b", sentence_str, re.IGNORECASE):
            entities.append({
                "text": dataset,
                "type": "Dataset"
            })

    for task in KNOWN_TASK_KEYWORDS:
        if re.search(rf"\b{re.escape(task)}\b", sentence_str, re.IGNORECASE):
            entities.append({
                "text": task,
                "type": "Task"
            })

    method_candidates = re.findall(
        r"\b[A-Z][A-Za-z0-9\-]*(?:Net|BERT|GPT|CNN|RNN|LSTM|Transformer)?\b",
        sentence_str
    )

    for method in method_candidates:
        if len(method) > 2 and method.lower() not in ["the", "this", "that", "using"]:
            entities.append({
                "text": method,
                "type": "Method"
            })

    unique_entities = []
    seen = set()

    for ent in entities:
        key = (ent["text"].lower(), ent["type"].lower())
        if key not in seen:
            unique_entities.append(ent)
            seen.add(key)

    return {
        "entities": unique_entities,
        "relations": relations
    }

In [ ]:
rule_df = test_df.copy()

rule_df["prediction_text"] = rule_df["sentence"].apply(
    lambda x: json.dumps(rule_based_extract(x), ensure_ascii=False, sort_keys=True)
)

rule_metrics = evaluate_structured_predictions(rule_df)
rule_metrics = add_equal_weighted_final_scores(rule_metrics)
rule_metrics = add_relation_weighted_final_scores(rule_metrics)
rule_metrics

In [ ]:
rule_ood_df = test_ood_df.copy()

rule_ood_df["prediction_text"] = rule_ood_df["sentence"].apply(
    lambda x: json.dumps(rule_based_extract(x), ensure_ascii=False, sort_keys=True)
)

rule_ood_metrics = evaluate_structured_predictions(rule_ood_df)
rule_ood_metrics = add_equal_weighted_final_scores(rule_ood_metrics)

rule_ood_metrics

In [ ]:
baseline_results = pd.DataFrame([

    {"model": "Rule-based baseline", "split": "test", **rule_metrics},
    {"model": "Rule-based baseline", "split": "test_ood", **rule_ood_metrics},
])

baseline_results

# FLAN-T5 Experiments
#FLAN-T5-small and FLAN-T5-base

###1.Zero-shot FLAN-T5-small




In [ ]:
!pip install -q transformers sentencepiece accelerate

In [ ]:
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

In [ ]:
model_name = "google/flan-t5-small"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
def make_zero_shot_prompt(sentence):
    prompt = f"""
Extract scientific entities and relations from the sentence.

Return only valid JSON with exactly this structure:
{{
  "entities": [
    {{"text": "...", "type": "..."}}
  ],
  "relations": [
    {{"head": "...", "relation": "...", "tail": "..."}}
  ]
}}

Entity types can include: Method, Task, Dataset, Metric, Material, Other.
Relation types can include: Used-For, Part-Of, Feature-Of, Compare, Evaluate-For, Conjunction, Hyponym-Of.

Sentence:
{sentence}

JSON:
"""
    return prompt.strip()

In [ ]:
def generate_flan_t5(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=4,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
zero_test_sample = test_df.head(20).copy()
predictions = []

for _, row in tqdm(zero_test_sample.iterrows(), total=len(zero_test_sample)):
    prompt = make_zero_shot_prompt(row["sentence"])
    pred_text = generate_flan_t5(prompt)

    predictions.append(pred_text)

zero_test_sample["prediction_text"] = predictions

In [ ]:
for i in range(5):
    print("SENTENCE:")
    print(zero_test_sample.iloc[i]["sentence"])

    print("\nGOLD:")
    print(zero_test_sample.iloc[i]["target_json"])

    print("\nPREDICTION:")
    print(zero_test_sample.iloc[i]["prediction_text"])
    print("=" * 100)

In [ ]:
zero_shot_test_metrics = evaluate_structured_predictions(zero_test_sample)
zero_shot_test_metrics = add_equal_weighted_final_scores(zero_shot_test_metrics)
zero_shot_test_metrics = add_relation_weighted_final_scores(zero_shot_test_metrics)
zero_shot_test_metrics

In [ ]:
zero_shot_test_metrics["valid_json_rate"]

###2.Few-shot FLAN-T5-small

In [ ]:
def make_strong_few_shot_json_prompt(sentence):
    prompt = f"""
You are a scientific information extraction system.

Your task:
Extract scientific entities and relations from the given sentence and return ONLY valid JSON.

Important output rules:
- Return only JSON.
- Do not explain anything.
- Do not use markdown.
- Do not add text before or after the JSON.
- Use double quotes for all keys and string values.
- Copy entity text exactly as it appears in the sentence.
- Every relation head and tail must be one of the extracted entity texts.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].

Allowed entity types:
1. Method
   - Models, algorithms, architectures, systems, techniques, mechanisms, approaches.
   - Examples: CornerNet, BERT, Transformer, convolution neural network, attention mechanism.

2. Task
   - NLP/CV/scientific tasks or problems.
   - Examples: object detection, machine translation, sentiment analysis, classification.

3. Dataset
   - Dataset or benchmark names.
   - Examples: MS COCO, ImageNet, SQuAD, MNIST.

4. Metric
   - Evaluation measures or scores.
   - Examples: AP, accuracy, F1, BLEU, ROUGE.

5. Material
   - Physical/scientific material, if mentioned.

6. Other
   - Scientific entity that does not clearly fit the above types.

Allowed relation types:
1. Used-For
   - A method is used for a task.
   - Example: CornerNet Used-For object detection.

2. Part-Of
   - A component is part of a method/system.
   - Example: convolution neural network Part-Of CornerNet.

3. Feature-Of
   - A feature/property belongs to a method or system.

4. Evaluated-With
   - A method is evaluated on a dataset or benchmark.
   - Example: CornerNet Evaluated-With MS COCO.

5. Evaluate-For
   - A metric/dataset is used to evaluate a task or method.

6. Compare-With
   - Two methods/systems are compared.

7. SubClass-Of
   - One method/entity is a subclass or type of another.

8. SubTask-Of
   - One task is a subtask of another task.

9. Synonym-Of
   - Two names refer to the same entity.
   - Example: ConvNets Synonym-Of convolutional neural networks.

10. Hyponym-Of
   - One entity is a more specific form of another.

11. Conjunction
   - Two entities are mentioned as coordinated/parallel items.

Required JSON format:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "one allowed entity type"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Example 1

Input sentence:
We propose CornerNet , a new approach to object detection where we detect an object bounding box as a pair of keypoints , the top - left corner and the bottom - right corner , using a single convolution neural network .

Output JSON:
{{
  "entities": [
    {{"text": "CornerNet", "type": "Method"}},
    {{"text": "object detection", "type": "Task"}},
    {{"text": "convolution neural network", "type": "Method"}}
  ],
  "relations": [
    {{"head": "convolution neural network", "relation": "Part-Of", "tail": "CornerNet"}},
    {{"head": "CornerNet", "relation": "Used-For", "tail": "object detection"}}
  ]
}}

Example 2

Input sentence:
Experiments show that CornerNet achieves a 4 2 . 2 % AP on MS COCO , outperforming all existing one - stage detectors .

Output JSON:
{{
  "entities": [
    {{"text": "CornerNet", "type": "Method"}},
    {{"text": "AP", "type": "Metric"}},
    {{"text": "MS COCO", "type": "Dataset"}}
  ],
  "relations": [
    {{"head": "CornerNet", "relation": "Evaluated-With", "tail": "MS COCO"}}
  ]
}}

Example 3

Input sentence:
Object detectors based on convolutional neural networks ( ConvNets ) have achieved state - of - the - art results on various challenging benchmarks .

Output JSON:
{{
  "entities": [
    {{"text": "convolutional neural networks", "type": "Method"}},
    {{"text": "ConvNets", "type": "Method"}}
  ],
  "relations": [
    {{"head": "ConvNets", "relation": "Synonym-Of", "tail": "convolutional neural networks"}}
  ]
}}

Example 4

Input sentence:
In this paper we introduce CornerNet , a new onestage approach to object detection that does away with anchor boxes .

Output JSON:
{{
  "entities": [
    {{"text": "CornerNet", "type": "Method"}},
    {{"text": "object detection", "type": "Task"}}
  ],
  "relations": [
    {{"head": "CornerNet", "relation": "Used-For", "tail": "object detection"}}
  ]
}}

Now extract from this sentence:
{sentence}

Output JSON:
"""
    return prompt.strip()

In [ ]:
import json

def extract_json_substring(text):
    """
    Extract the part between the first { and the last }.
    Useful when the model generates extra text around JSON.
    """
    text = str(text).strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text


def repair_prediction_text(text):
    """
    Try to repair model output by extracting the JSON part.
    If it is valid JSON, return normalized JSON string.
    Otherwise return the original candidate.
    """
    candidate = extract_json_substring(text)

    try:
        obj = json.loads(candidate)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        return candidate

In [ ]:
strong_prompt_test_df = test_df.copy()

raw_predictions = []

for _, row in tqdm(strong_prompt_test_df.iterrows(), total=len(strong_prompt_test_df)):
    prompt = make_strong_few_shot_json_prompt(row["sentence"])
    raw_output = generate_flan_t5(prompt)
    raw_predictions.append(raw_output)

strong_prompt_test_df["prediction_text_raw"] = raw_predictions


In [ ]:
strong_prompt_test_df["prediction_text"] = strong_prompt_test_df["prediction_text_raw"].apply(repair_prediction_text)

In [ ]:
strong_prompt_metrics = evaluate_structured_predictions(strong_prompt_test_df)

strong_prompt_metrics = add_equal_weighted_final_scores(strong_prompt_metrics)
strong_prompt_metrics = add_relation_weighted_final_scores(strong_prompt_metrics)
strong_prompt_metrics = add_partial_entity_recall(strong_prompt_metrics, strong_prompt_test_df)

strong_prompt_metrics

In [ ]:
strong_prompt_result = pd.DataFrame([
    {
        "model": "Few-shot FLAN-T5-small: Strong JSON Prompt",
        "split": "test",
        **strong_prompt_metrics
    }
])

strong_prompt_result

# Few-shot FLAN-T5 with Label-Constrained Prompting and Flexible Parsing

1. Uses FLAN-T5 in few-shot mode
2. Adds allowed entity/relation labels from training data into the prompt
3. Converts imperfect model output into JSON using a flexible parser

In [ ]:
from collections import Counter
import json
import re
import difflib

entity_type_counter = Counter()
relation_type_counter = Counter()

for types in train_df["entity_types"]:
    entity_type_counter.update(types)

for types in train_df["relation_types"]:
    relation_type_counter.update(types)

ALLOWED_ENTITY_TYPES = sorted(entity_type_counter.keys())
ALLOWED_RELATION_TYPES = sorted(relation_type_counter.keys())

print("Entity types:", ALLOWED_ENTITY_TYPES)
print("Relation types:", ALLOWED_RELATION_TYPES)

In [ ]:
def make_10shot_prompt(sentence, n_examples=10):
    examples_text = []

    for i in range(n_examples):
        example_sentence = train_df.iloc[i]["sentence"]
        example_target = json.loads(train_df.iloc[i]["target_json"])

        example_entities = example_target["entities"]
        example_relations = example_target["relations"]

        entity_lines = "\n".join([
            f"ENTITY: {e['text']} | {e['type']}"
            for e in example_entities
        ])

        relation_lines = "\n".join([
            f"RELATION: {r['head']} | {r['relation']} | {r['tail']}"
            for r in example_relations
        ])

        if not entity_lines:
            entity_lines = "NO_ENTITIES"

        if not relation_lines:
            relation_lines = "NO_RELATIONS"

        example_block = f"""
Example {i + 1}

Sentence:
{example_sentence}

Output:
{entity_lines}
{relation_lines}
"""
        examples_text.append(example_block.strip())

    examples_text = "\n\n".join(examples_text)

    prompt = f"""
You are a scientific information extraction system.

Extract scientific entities and relations from the sentence.

Allowed entity types:
{", ".join(ALLOWED_ENTITY_TYPES)}

Allowed relation types:
{", ".join(ALLOWED_RELATION_TYPES)}

Output rules:
- Copy entity text exactly from the sentence.
- Use only the allowed entity types.
- Use only the allowed relation types.
- Do not explain anything.
- Do not write JSON.
- Use only this line format:

ENTITY: entity text | entity type
RELATION: head entity | relation type | tail entity

If there are no entities, write:
NO_ENTITIES

If there are no relations, write:
NO_RELATIONS

Here are examples:

{examples_text}

Now extract from this sentence:

Sentence:
{sentence}

Output:
"""
    return prompt.strip()

In [ ]:
def generate_flan_t5(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
            repetition_penalty=1.8,
            no_repeat_ngram_size=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
import json
import re
import difflib

def closest_allowed_label(label, allowed_labels):
    label = str(label).strip()

    if label in allowed_labels:
        return label

    matches = difflib.get_close_matches(label, allowed_labels, n=1, cutoff=0.6)

    if matches:
        return matches[0]

    return None


def clean_extracted_entity(text):
    text = str(text).strip()

    # Remove parenthetical hints like (model), (Method), etc.
    text = re.sub(r"\([^)]*\)", "", text)

    # Remove quotes and normalize spaces
    text = text.replace('"', "").replace("'", "")
    text = re.sub(r"\s+", " ", text)

    return text.strip(" ,;:.-")


def infer_entity_type(entity_text):
    """
    Guess entity type if the model does not provide one.
    This is post-processing, not gold labeling.
    """
    t = str(entity_text).lower()

    dataset_keywords = [
        "imagenet", "coco", "ms coco", "squad", "mnist", "cifar", "glue"
    ]

    metric_keywords = [
        "accuracy", "f1", "bleu", "rouge", "precision", "recall", "ap"
    ]

    task_keywords = [
        "classification", "detection", "object detection", "translation",
        "machine translation", "summarization", "sentiment analysis",
        "entailment", "question answering", "nlp", "natural language processing",
        "sequence - to - sequence", "sequence-to-sequence"
    ]

    method_keywords = [
        "bert", "transformer", "network", "networks", "model", "models",
        "architecture", "architectures", "attention", "mechanism",
        "encoder", "decoder", "cnn", "rnn", "lstm", "neural",
        "feature extraction", "cornernet", "convnet", "convnets"
    ]

    if any(k in t for k in dataset_keywords):
        return "Dataset"

    if any(k in t for k in metric_keywords):
        return "Metric"

    if any(k in t for k in task_keywords):
        return "Task"

    if any(k in t for k in method_keywords):
        return "Method"

    return "Other"


def normalize_for_match(text):
    text = str(text).lower()
    text = re.sub(r"[^a-z0-9]+", "", text)
    return text


def entity_appears_in_sentence(entity_text, sentence):
    """
    Keep only entities that appear in the sentence approximately.
    """
    entity_text = str(entity_text).strip()
    sentence = str(sentence)

    if not entity_text:
        return False

    if entity_text.lower() in sentence.lower():
        return True

    compact_entity = normalize_for_match(entity_text)
    compact_sentence = normalize_for_match(sentence)

    if not compact_entity:
        return False

    return compact_entity in compact_sentence


def split_entity_candidates(content):
    """
    Split messy model output into possible entity candidates.
    Handles commas, semicolons, and repeated outputs.
    """
    content = str(content)

    # Remove repeated ENTITY prefix if model repeats it
    content = re.sub(r"(?i)entity\s*:", "", content)

    # Split on common separators
    candidates = re.split(r";|,|\band\b", content)

    cleaned = []
    for cand in candidates:
        cand = clean_extracted_entity(cand)
        if cand:
            cleaned.append(cand)

    return cleaned


def parse_better_line_based_output_to_json(raw_output, sentence):
    entities = []
    relations = []

    seen_entities = set()
    seen_relations = set()

    lines = str(raw_output).splitlines()

    for line in lines:
        line = line.strip()

        if not line:
            continue

        # Parse ENTITY lines

        if line.lower().startswith("entity:"):
            content = line[len("ENTITY:"):].strip()

            # Case 1: ideal format -> ENTITY: entity text | entity type
            if "|" in content:
                parts = [p.strip() for p in content.split("|")]

                entity_text = clean_extracted_entity(parts[0])
                entity_type = parts[1].strip() if len(parts) > 1 else None

                entity_type = closest_allowed_label(entity_type, ALLOWED_ENTITY_TYPES)

                if entity_type is None:
                    entity_type = infer_entity_type(entity_text)

                if entity_text and entity_appears_in_sentence(entity_text, sentence):
                    key = (entity_text.lower(), entity_type.lower())

                    if key not in seen_entities:
                        entities.append({
                            "text": entity_text,
                            "type": entity_type
                        })
                        seen_entities.add(key)

            # Case 2: loose format -> ENTITY: BERT, Transformer, attention mechanism
            else:
                candidates = split_entity_candidates(content)

                for entity_text in candidates:
                    if not entity_appears_in_sentence(entity_text, sentence):
                        continue

                    entity_type = infer_entity_type(entity_text)

                    key = (entity_text.lower(), entity_type.lower())

                    if key not in seen_entities:
                        entities.append({
                            "text": entity_text,
                            "type": entity_type
                        })
                        seen_entities.add(key)

        # Parse RELATION lines

        elif line.lower().startswith("relation:"):
            content = line[len("RELATION:"):].strip()

            # Ideal format -> RELATION: head | relation | tail
            if "|" in content:
                parts = [p.strip() for p in content.split("|")]

                if len(parts) >= 3:
                    head = clean_extracted_entity(parts[0])
                    relation = closest_allowed_label(parts[1], ALLOWED_RELATION_TYPES)
                    tail = clean_extracted_entity(parts[2])

                    if head and relation and tail:
                        key = (head.lower(), relation.lower(), tail.lower())

                        if key not in seen_relations:
                            relations.append({
                                "head": head,
                                "relation": relation,
                                "tail": tail
                            })
                            seen_relations.add(key)

    return json.dumps(
        {
            "entities": entities,
            "relations": relations
        },
        ensure_ascii=False,
        sort_keys=True
    )

In [ ]:
from tqdm.auto import tqdm
!pip install -q transformers sentencepiece accelerate

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-small"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
few_10shot_test_df = test_df.copy()

raw_predictions = []
json_predictions = []

for _, row in tqdm(few_10shot_test_df.iterrows(), total=len(few_10shot_test_df)):
    prompt = make_10shot_prompt(row["sentence"], n_examples=10)
    raw_output = generate_flan_t5(prompt)

    json_output = parse_better_line_based_output_to_json(
        raw_output,
        row["sentence"]
    )

    raw_predictions.append(raw_output)
    json_predictions.append(json_output)

few_10shot_test_df["prediction_text_raw"] = raw_predictions
few_10shot_test_df["prediction_text"] = json_predictions

In [ ]:
few_10shot_metrics = evaluate_structured_predictions(few_10shot_test_df)

few_10shot_metrics = add_equal_weighted_final_scores(few_10shot_metrics)
few_10shot_metrics = add_relation_weighted_final_scores(few_10shot_metrics)
few_10shot_metrics = add_partial_entity_recall(few_10shot_metrics, few_10shot_test_df)

few_10shot_metrics

In [ ]:
few_10shot_result = pd.DataFrame([
    {
        "model": "10-shot FLAN-T5-small + Line Parser",
        "split": "test",
        **few_10shot_metrics
    }
])

few_10shot_result

In [ ]:
output_dir = "/content/drive/MyDrive/NLP_26_Project/SciER_outputs"


In [ ]:
few_10shot_test_df.to_json(
    f"{output_dir}/few_10shot_flan_t5_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
few_10shot_result.to_csv(
    f"{output_dir}/few_10shot_flan_t5_test_metrics.csv",
    index=False
)

In [ ]:
!ls -lh "/content/drive/MyDrive/NLP_26_Project/SciER_outputs"

#FLAN-T5-base zero-shot

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

model_name = "google/flan-t5-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device)

In [ ]:
def make_zero_shot_line_prompt(sentence):
    prompt = f"""
You are a scientific information extraction system.

Extract scientific entities and relations from the sentence.

Allowed entity types:
{", ".join(ALLOWED_ENTITY_TYPES)}

Allowed relation types:
{", ".join(ALLOWED_RELATION_TYPES)}

Output rules:
- Copy entity text exactly from the sentence.
- Use only the allowed entity types.
- Use only the allowed relation types.
- Do not explain anything.
- Do not write JSON.
- Use only this line format:

ENTITY: entity text | entity type
RELATION: head entity | relation type | tail entity

If there are no entities, write:
NO_ENTITIES

If there are no relations, write:
NO_RELATIONS

Sentence:
{sentence}

Output:
"""
    return prompt.strip()

In [ ]:
def generate_flan_t5(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            num_beams=1,
            do_sample=False,
            repetition_penalty=1.5,
            no_repeat_ngram_size=4,
            early_stopping=True
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [ ]:
flan_base_zero_test_df = test_df.copy()

raw_predictions = []
json_predictions = []

for _, row in tqdm(flan_base_zero_test_df.iterrows(), total=len(flan_base_zero_test_df)):
    prompt = make_zero_shot_line_prompt(row["sentence"])
    raw_output = generate_flan_t5(prompt)

    json_output = parse_better_line_based_output_to_json(
        raw_output,
        row["sentence"]
    )

    raw_predictions.append(raw_output)
    json_predictions.append(json_output)

flan_base_zero_test_df["prediction_text_raw"] = raw_predictions
flan_base_zero_test_df["prediction_text"] = json_predictions

In [ ]:
for i in range(5):
    print("SENTENCE:")
    print(flan_base_zero_test_df.iloc[i]["sentence"])

    print("\nRAW OUTPUT:")
    print(flan_base_zero_test_df.iloc[i]["prediction_text_raw"])

    print("\nCONVERTED JSON:")
    print(flan_base_zero_test_df.iloc[i]["prediction_text"])

    print("=" * 100)

In [ ]:
def partial_entity_match(gold_entity, pred_entity):
    """
    Returns True if predicted entity partially matches gold entity
    and the entity type is the same.
    """
    gold_text, gold_type = gold_entity
    pred_text, pred_type = pred_entity

    if gold_type != pred_type:
        return False

    gold_text = str(gold_text).lower().strip()
    pred_text = str(pred_text).lower().strip()

    return (
        gold_text in pred_text
        or pred_text in gold_text
    )

In [ ]:
# Evaluate FLAN-T5-base zero-shot predictions
flan_base_zero_metrics = evaluate_structured_predictions(flan_base_zero_test_df)

flan_base_zero_metrics = add_equal_weighted_final_scores(flan_base_zero_metrics)
flan_base_zero_metrics = add_relation_weighted_final_scores(flan_base_zero_metrics)
flan_base_zero_metrics = add_partial_entity_recall(
    flan_base_zero_metrics,
    flan_base_zero_test_df
)

flan_base_zero_metrics

In [ ]:
flan_base_zero_result = pd.DataFrame([
    {
        "model": "Zero-shot FLAN-T5-base + Label-Constrained Prompt + Flexible Parser",
        "split": "test",
        **flan_base_zero_metrics
    }
])

flan_base_zero_result.to_csv(
    f"{output_dir}/flan_t5_base_zero_shot_test_metrics.csv",
    index=False
)

flan_base_zero_test_df.to_json(
    f"{output_dir}/flan_t5_base_zero_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

flan_base_zero_result

In [ ]:
flan_base_zero_test_df.to_json(
    f"{output_dir}/flan_t5_base_zero_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

In [ ]:
flan_base_zero_result.to_csv(
    f"{output_dir}/flan_t5_base_zero_shot_test_metrics.csv",
    index=False
)

In [ ]:
comparison_results = flan_base_zero_result.copy()

comparison_results.to_csv(
    f"{output_dir}/comparison_results.csv",
    index=False
)

comparison_results

## Few-shot FLAN-T5-base: Direct JSON Generation

In [ ]:

def make_few_shot_json_prompt(sentence, n_examples=3):
    examples_text = []

    for i in range(n_examples):
        example_sentence = train_df.iloc[i]["sentence"]
        example_json = train_df.iloc[i]["target_json"]

        example_block = f"""
Example {i + 1}

Sentence:
{example_sentence}

JSON:
{example_json}
"""
        examples_text.append(example_block.strip())

    examples_text = "\n\n".join(examples_text)

    prompt = f"""
You are a scientific information extraction system.

Extract scientific entities and relations from the sentence.

Allowed entity types:
{", ".join(ALLOWED_ENTITY_TYPES)}

Allowed relation types:
{", ".join(ALLOWED_RELATION_TYPES)}

Return ONLY valid JSON.
Do not explain anything.
Do not use markdown.
Do not add text before or after the JSON.

The JSON must have exactly this structure:
{{
  "entities": [
    {{"text": "exact entity span", "type": "entity type"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only the allowed entity types.
- Use only the allowed relation types.
- Every relation head and tail must appear in the entities list.
- If there are no entities, use "entities": [].
- If there are no relations, use "relations": [].

Examples:

{examples_text}

Now extract from this sentence:

Sentence:
{sentence}

JSON:
"""
    return prompt.strip()

In [ ]:
flan_base_fewshot_json_test_df = test_df.copy()

raw_predictions = []

for _, row in tqdm(flan_base_fewshot_json_test_df.iterrows(), total=len(flan_base_fewshot_json_test_df)):
    prompt = make_few_shot_json_prompt(row["sentence"], n_examples=3)
    raw_output = generate_flan_t5(prompt)

    raw_predictions.append(raw_output)

flan_base_fewshot_json_test_df["prediction_text"] = raw_predictions

In [ ]:
for i in range(5):
    print("SENTENCE:")
    print(flan_base_fewshot_json_test_df.iloc[i]["sentence"])

    print("\nGOLD JSON:")
    print(flan_base_fewshot_json_test_df.iloc[i]["target_json"])

    print("\nMODEL OUTPUT:")
    print(flan_base_fewshot_json_test_df.iloc[i]["prediction_text"])

    print("=" * 100)

In [ ]:
flan_base_fewshot_json_metrics = evaluate_structured_predictions(
    flan_base_fewshot_json_test_df
)

flan_base_fewshot_json_metrics = add_equal_weighted_final_scores(
    flan_base_fewshot_json_metrics
)

flan_base_fewshot_json_metrics = add_relation_weighted_final_scores(
    flan_base_fewshot_json_metrics
)

flan_base_fewshot_json_metrics = add_partial_entity_recall(
    flan_base_fewshot_json_metrics,
    flan_base_fewshot_json_test_df
)

flan_base_fewshot_json_metrics

In [ ]:
flan_base_fewshot_json_result = pd.DataFrame([
    {
        "model": "Few-shot FLAN-T5-base: Direct JSON Generation",
        "split": "test",
        **flan_base_fewshot_json_metrics
    }
])

flan_base_fewshot_json_result

In [ ]:
output_dir = "/content/drive/MyDrive/NLP_26_Project/SciER_outputs"

flan_base_fewshot_json_test_df.to_json(
    f"{output_dir}/flan_t5_base_few_shot_direct_json_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

flan_base_fewshot_json_result.to_csv(
    f"{output_dir}/flan_t5_base_few_shot_direct_json_test_metrics.csv",
    index=False
)

In [ ]:
comparison_results = pd.concat(
    [
        comparison_results,
        flan_base_fewshot_json_result
    ],
    ignore_index=True
)

comparison_results

#Fine Tuning FLAN-T5-base

In [ ]:


!pip install -q transformers datasets accelerate sentencepiece

import os
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainingArguments,
    Seq2SeqTrainer
)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

model_name = "google/flan-t5-base"

tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

def build_input_text(sentence):
    return "Extract scientific entities and relations as JSON: " + str(sentence)

# Keep original dataframes safe
train_data = train_df.copy()
dev_data = dev_df.copy()
test_data = test_df.copy()

train_data["input_text"] = train_data["sentence"].apply(build_input_text)
dev_data["input_text"] = dev_data["sentence"].apply(build_input_text)
test_data["input_text"] = test_data["sentence"].apply(build_input_text)

train_data = train_data[["input_text", "target_json"]]
dev_data = dev_data[["input_text", "target_json"]]
test_data = test_data[["input_text", "target_json"]]

# Start small for testing
train_small = train_data.sample(n=min(1000, len(train_data)), random_state=42).reset_index(drop=True)
dev_small = dev_data.sample(n=min(200, len(dev_data)), random_state=42).reset_index(drop=True)
test_small = test_data.sample(n=min(200, len(test_data)), random_state=42).reset_index(drop=True)

print("Train:", len(train_small))
print("Dev:", len(dev_small))
print("Test:", len(test_small))

train_dataset = Dataset.from_pandas(train_small)
dev_dataset = Dataset.from_pandas(dev_small)
test_dataset = Dataset.from_pandas(test_small)

max_input_length = 256
max_target_length = 256

def preprocess_function(batch):
    model_inputs = tokenizer(
        batch["input_text"],
        max_length=max_input_length,
        truncation=True
    )

    labels = tokenizer(
        text_target=batch["target_json"],
        max_length=max_target_length,
        truncation=True
    )

    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_train = train_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_dev = dev_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=dev_dataset.column_names
)

tokenized_test = test_dataset.map(
    preprocess_function,
    batched=True,
    remove_columns=test_dataset.column_names
)

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tokenizer,
    model=model
)

In [ ]:


for param in model.parameters():
    param.requires_grad = False

# Unfreeze last encoder block
for param in model.encoder.block[-1].parameters():
    param.requires_grad = True

# Unfreeze last decoder block
for param in model.decoder.block[-1].parameters():
    param.requires_grad = True

# Unfreeze final layer norms
for param in model.encoder.final_layer_norm.parameters():
    param.requires_grad = True

for param in model.decoder.final_layer_norm.parameters():
    param.requires_grad = True

# Unfreeze language modeling head
for param in model.lm_head.parameters():
    param.requires_grad = True

model = model.to(device)

total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("Total parameters:", total_params)
print("Trainable parameters:", trainable_params)
print("Trainable percentage:", round(100 * trainable_params / total_params, 4), "%")

In [ ]:
for i in range(3):
    print("INPUT:")
    print(train_small.iloc[i]["input_text"])
    print("\nTARGET:")
    print(train_small.iloc[i]["target_json"])
    print("=" * 100)

In [ ]:


import os
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer

output_dir = "/content/drive/MyDrive/NLP_26_Project/flan_t5_base_partial_finetuned_scier"
os.makedirs(output_dir, exist_ok=True)

training_args = Seq2SeqTrainingArguments(
    output_dir=output_dir,

    eval_strategy="steps",
    eval_steps=100,
    save_steps=100,
    logging_steps=25,

    learning_rate=1e-3,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,

    num_train_epochs=3,

    predict_with_generate=True,
    generation_max_length=256,

    save_total_limit=2,
    load_best_model_at_end=True,

    fp16=torch.cuda.is_available(),

    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator
)

trainer.train()

trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("Partial fine-tuned model saved to:", output_dir)

In [ ]:

model.eval()

def generate_prediction(input_text):
    inputs = tokenizer(
        input_text,
        return_tensors="pt",
        truncation=True,
        max_length=max_input_length
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=256,
            num_beams=4,
            do_sample=False
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

partial_finetuned_test_df = test_small.copy()

predictions = []

for text in tqdm(partial_finetuned_test_df["input_text"], total=len(partial_finetuned_test_df)):
    pred = generate_prediction(text)
    predictions.append(pred)

partial_finetuned_test_df["prediction_text"] = predictions

# Inspect examples
for i in range(5):
    print("INPUT:")
    print(partial_finetuned_test_df.iloc[i]["input_text"])

    print("\nGOLD:")
    print(partial_finetuned_test_df.iloc[i]["target_json"])

    print("\nPREDICTION:")
    print(partial_finetuned_test_df.iloc[i]["prediction_text"])

    print("=" * 100)

# Evaluate with your existing functions
partial_finetuned_metrics = evaluate_structured_predictions(partial_finetuned_test_df)

partial_finetuned_metrics = add_equal_weighted_final_scores(partial_finetuned_metrics)
partial_finetuned_metrics = add_relation_weighted_final_scores(partial_finetuned_metrics)
partial_finetuned_metrics = add_partial_entity_recall(
    partial_finetuned_metrics,
    partial_finetuned_test_df
)

partial_finetuned_metrics

In [ ]:
# Save predictions and metrics

output_dir_results = "/content/drive/MyDrive/NLP_26_Project/SciER_outputs"
os.makedirs(output_dir_results, exist_ok=True)

partial_finetuned_result = pd.DataFrame([
    {
        "model": "Partial fine-tuned FLAN-T5-base: last decoder block + lm_head",
        "split": "test_200",
        **partial_finetuned_metrics
    }
])

partial_finetuned_test_df.to_json(
    f"{output_dir_results}/partial_finetuned_flan_t5_base_test_200_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

partial_finetuned_result.to_csv(
    f"{output_dir_results}/partial_finetuned_flan_t5_base_test_200_metrics.csv",
    index=False
)

partial_finetuned_result

# Qwen Experiments

### Qwen2.5-1.5B-Instruct: Zero-shot Direct JSON Generation


In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
!pip install -q transformers accelerate bitsandbytes

In [ ]:
import torch
import json
import pandas as pd
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Model loaded:", model_name)

In [ ]:
def make_qwen_zero_shot_prompt(sentence):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific information extraction system. "
                "Your job is to extract scientific entities and relations from a sentence. "
                "Return only valid JSON. Do not explain anything."
            )
        },
        {
            "role": "user",
            "content": f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method: models, algorithms, architectures, systems, techniques, mechanisms.
- Task: NLP/CV/scientific tasks or problems.
- Dataset: dataset or benchmark names.
- Metric: evaluation metrics or scores.
- Material: physical/scientific materials.
- Other: scientific entity that does not fit the above.

Allowed relation types:
- Used-For
- Part-Of
- Feature-Of
- Evaluated-With
- Evaluate-For
- Compare-With
- SubClass-Of
- SubTask-Of
- Synonym-Of
- Hyponym-Of
- Conjunction

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "entity type"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only the allowed entity types.
- Use only the allowed relation types.
- Every relation head and tail must appear in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON, with no extra text.

Sentence:
{sentence}
"""
        }
    ]

    prompt = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    return prompt

In [ ]:
def generate_qwen(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return prediction.strip()

In [ ]:
def extract_json_substring(text):
    text = str(text).strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text


def repair_prediction_text(text):
    candidate = extract_json_substring(text)

    try:
        obj = json.loads(candidate)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        return candidate

In [ ]:
qwen_zero_test20_df = test_df.head(20).copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(qwen_zero_test20_df.iterrows(), total=len(qwen_zero_test20_df)):
    prompt = make_qwen_zero_shot_prompt(row["sentence"])
    raw_output = generate_qwen(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

qwen_zero_test20_df["prediction_text_raw"] = raw_predictions
qwen_zero_test20_df["prediction_text"] = repaired_predictions

In [ ]:
for i in range(5):
    print("SENTENCE:")
    print(qwen_zero_test20_df.iloc[i]["sentence"])

    print("\nGOLD:")
    print(qwen_zero_test20_df.iloc[i]["target_json"])

    print("\nRAW QWEN OUTPUT:")
    print(qwen_zero_test20_df.iloc[i]["prediction_text_raw"])

    print("\nREPAIRED OUTPUT:")
    print(qwen_zero_test20_df.iloc[i]["prediction_text"])

    print("=" * 100)

In [ ]:
def partial_entity_match(gold_entity, pred_entity):
    """
    Partial match between gold and predicted entities.
    The entity type must be the same.
    The entity text can be partially overlapping.
    """
    gold_text, gold_type = gold_entity
    pred_text, pred_type = pred_entity

    gold_text = str(gold_text).lower().strip()
    pred_text = str(pred_text).lower().strip()
    gold_type = str(gold_type).lower().strip()
    pred_type = str(pred_type).lower().strip()

    if gold_type != pred_type:
        return False

    return (
        gold_text in pred_text
        or pred_text in gold_text
    )

In [ ]:
qwen_zero_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(qwen_zero_test_df.iterrows(), total=len(qwen_zero_test_df)):
    prompt = make_qwen_zero_shot_prompt(row["sentence"])
    raw_output = generate_qwen(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

qwen_zero_test_df["prediction_text_raw"] = raw_predictions
qwen_zero_test_df["prediction_text"] = repaired_predictions

In [ ]:
qwen_zero_metrics = evaluate_structured_predictions(qwen_zero_test_df)

qwen_zero_metrics = add_equal_weighted_final_scores(qwen_zero_metrics)
qwen_zero_metrics = add_relation_weighted_final_scores(qwen_zero_metrics)
qwen_zero_metrics = add_partial_entity_recall(
    qwen_zero_metrics,
    qwen_zero_test_df
)

qwen_zero_metrics

#Qwen Few-shot Direct JSON Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Loaded:", model_name)

In [ ]:
import json
import torch
from tqdm.auto import tqdm

def extract_json_substring(text):
    text = str(text).strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text


def repair_prediction_text(text):
    candidate = extract_json_substring(text)

    try:
        obj = json.loads(candidate)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        return candidate


def generate_qwen(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
def make_qwen_few_shot_prompt(sentence, n_examples=3):
    examples_text = []

    for i in range(n_examples):
        example_sentence = train_df.iloc[i]["sentence"]
        example_json = train_df.iloc[i]["target_json"]

        example_block = f"""
Example {i + 1}

Sentence:
{example_sentence}

Output JSON:
{example_json}
"""
        examples_text.append(example_block.strip())

    examples_text = "\n\n".join(examples_text)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific information extraction system. "
                "Extract scientific entities and relations from scientific sentences. "
                "Return only valid JSON. Do not explain anything."
            )
        },
        {
            "role": "user",
            "content": f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method: models, algorithms, architectures, systems, techniques, mechanisms, embeddings, neural networks, classifiers, detectors, or scientific approaches.
- Task: scientific/NLP/CV tasks or problems, such as classification, object detection, machine translation, sentiment analysis, entailment, question answering, or text classification.
- Dataset: dataset or benchmark names, such as MS COCO, ImageNet, BioASQ, SQuAD, MNIST, CIFAR, or other named datasets.

Allowed relation types:
- Part-Of: one entity is a component or part of another entity.
- Used-For: a Method is used for a Task, or a scientific approach is applied to solve a task.
- Evaluated-With: a Method is evaluated using a Dataset or benchmark.
- Synonym-Of: two entity names refer to the same concept or abbreviation.
- Compare-With: two Methods, systems, or approaches are compared.
- SubClass-Of: one entity is a specific type/subclass of another entity.
- Benchmark-For: a Dataset or benchmark is used for evaluating a Task or Method.
- SubTask-Of: one Task is a smaller task under a broader Task.
- Trained-With: a Method is trained using a Dataset or resource.

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only these entity types: Method, Task, Dataset.
- Use only these relation types: Part-Of, Used-For, Evaluated-With, Synonym-Of, Compare-With, SubClass-Of, Benchmark-For, SubTask-Of, Trained-With.
- Do not invent entities that are not explicitly mentioned in the sentence.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON, with no extra text.

Few-shot examples:

{examples_text}

Now extract from this sentence:

Sentence:
{sentence}

Output JSON:
"""
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
qwen_few_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(qwen_few_test_df.iterrows(), total=len(qwen_few_test_df)):
    prompt = make_qwen_few_shot_prompt(row["sentence"], n_examples=3)
    raw_output = generate_qwen(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

qwen_few_test_df["prediction_text_raw"] = raw_predictions
qwen_few_test_df["prediction_text"] = repaired_predictions

In [ ]:
def fix_invalid_entity_types(pred_text):
    obj, valid = parse_json_safe(pred_text)

    if not valid:
        return pred_text

    allowed_types = {"Method", "Task", "Dataset"}

    for ent in obj.get("entities", []):
        if ent.get("type") not in allowed_types:
            ent["type"] = "Method"

    return json.dumps(obj, ensure_ascii=False, sort_keys=True)


qwen_few_test_df["prediction_text"] = qwen_few_test_df["prediction_text"].apply(
    fix_invalid_entity_types
)

In [ ]:
qwen_few_metrics = evaluate_structured_predictions(qwen_few_test_df)

qwen_few_metrics = add_equal_weighted_final_scores(qwen_few_metrics)

qwen_few_metrics = add_relation_weighted_final_scores(
    qwen_few_metrics
)

qwen_few_metrics = add_partial_entity_recall(
    qwen_few_metrics,
    qwen_few_test_df
)

qwen_few_metrics

In [ ]:
import os

os.makedirs(output_dir, exist_ok=True)

qwen_few_result = pd.DataFrame([
    {
        "model": "Few-shot Qwen2.5-1.5B-Instruct: Direct JSON Generation",
        "split": "test",
        **qwen_few_metrics
    }
])

qwen_few_test_df.to_json(
    f"{output_dir}/qwen2_5_1_5b_few_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

qwen_few_result.to_csv(
    f"{output_dir}/qwen2_5_1_5b_few_shot_test_metrics.csv",
    index=False
)

qwen_few_result

In [ ]:
qwen_few_metrics = {
    "valid_json_rate": 0.955503512880562,
    "exact_match_rate": 0.0234192037470726,
    "entity_micro_precision": 0.46405579399141633,
    "entity_micro_recall": 0.3127259580621837,
    "entity_micro_f1": 0.37365010799136067,
    "entity_example_precision": 0.4482630757220921,
    "entity_example_recall": 0.3502167459134672,
    "entity_example_f1": 0.3796601930160272,
    "relation_micro_precision": 0.0512630014858841,
    "relation_micro_recall": 0.043588123815540114,
    "relation_micro_f1": 0.047115056333219524,
    "relation_example_precision": 0.13786104605776736,
    "relation_example_recall": 0.13653721051262036,
    "relation_example_f1": 0.13424323529709944,
    "final_micro_f1_equal_weighted": 0.2103825821622901,
    "final_example_f1_equal_weighted": 0.2569517141565633,
    "final_micro_f1_relation_weighted": 0.17772907699647597,
    "final_example_f1_relation_weighted": 0.23241001838467057,
    "partial_entity_recall": 0.44420590143658056
}

qwen_few_metrics

#Qwen2.5-1.5B-Instruct LoRA Fine-tuning

In [ ]:
!pip install -q transformers==4.51.3 accelerate==1.6.0 bitsandbytes==0.45.5 peft datasets

import os
import json
import re
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = """
You are a scientific information extraction system.
Extract scientific entities and relations from scientific sentences.
Return only valid JSON. Do not explain anything.
""".strip()


def build_user_prompt(sentence):
    return f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method: models, algorithms, architectures, systems, techniques, mechanisms, embeddings, neural networks, classifiers, detectors, or scientific approaches.
- Task: scientific/NLP/CV tasks or problems, such as classification, object detection, machine translation, sentiment analysis, entailment, question answering, or text classification.
- Dataset: dataset or benchmark names, such as MS COCO, ImageNet, BioASQ, SQuAD, MNIST, CIFAR, or other named datasets.

Allowed relation types:
- Part-Of: one entity is a component or part of another entity.
- Used-For: a Method is used for a Task, or a scientific approach is applied to solve a task.
- Evaluated-With: a Method is evaluated using a Dataset or benchmark.
- Synonym-Of: two entity names refer to the same concept or abbreviation.
- Compare-With: two Methods, systems, or approaches are compared.
- SubClass-Of: one entity is a specific type/subclass of another entity.
- Benchmark-For: a Dataset or benchmark is used for evaluating a Task or Method.
- SubTask-Of: one Task is a smaller task under a broader Task.
- Trained-With: a Method is trained using a Dataset or resource.

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only these entity types: Method, Task, Dataset.
- Use only these relation types: Part-Of, Used-For, Evaluated-With, Synonym-Of, Compare-With, SubClass-Of, Benchmark-For, SubTask-Of, Trained-With.
- Do not invent entities that are not explicitly mentioned in the sentence.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON, with no extra text.

Sentence:
{sentence}
""".strip()


def build_training_text(sentence, target_json):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(sentence)},
        {"role": "assistant", "content": target_json}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [ ]:
train_ft = train_df.sample(
    n=min(3000, len(train_df)),
    random_state=42
).reset_index(drop=True)

dev_ft = dev_df.sample(
    n=min(300, len(dev_df)),
    random_state=42
).reset_index(drop=True)

test_ft = test_df.copy().reset_index(drop=True)

print("Train fine-tuning size:", len(train_ft))
print("Dev size:", len(dev_ft))
print("Full test size:", len(test_ft))

train_ft["text"] = train_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

dev_ft["text"] = dev_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

train_dataset = Dataset.from_pandas(train_ft[["text"]])
dev_dataset = Dataset.from_pandas(dev_ft[["text"]])

print(train_ft["text"].iloc[0][:1200])

In [ ]:
max_length = 1024

def tokenize_function(batch):
    outputs = tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False
    )

    outputs["labels"] = outputs["input_ids"].copy()
    return outputs


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_dev = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dev_dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
qwen_lora_output_dir = f"{project_dir}/qwen_1_5b_lora_scier_train3000"
os.makedirs(qwen_lora_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=qwen_lora_output_dir,

    num_train_epochs=2,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,

    save_total_limit=2,
    load_best_model_at_end=True,

    fp16=True,
    report_to="none",

    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator
)

trainer.train()

model.save_pretrained(qwen_lora_output_dir)
tokenizer.save_pretrained(qwen_lora_output_dir)

print("LoRA adapter saved to:", qwen_lora_output_dir)

In [ ]:
model.eval()

def make_qwen_lora_prompt(sentence):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(sentence)}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_qwen_lora(sentence, max_new_tokens=256):
    prompt = make_qwen_lora_prompt(sentence)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return prediction.strip()


output_dir_results = output_dir
os.makedirs(output_dir_results, exist_ok=True)

qwen_lora_full_test_df = test_ft.copy()

raw_predictions = []
repaired_predictions = []

for idx, row in tqdm(qwen_lora_full_test_df.iterrows(), total=len(qwen_lora_full_test_df)):
    raw_output = generate_qwen_lora(row["sentence"])
    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

    if (idx + 1) % 100 == 0:
        temp_df = qwen_lora_full_test_df.iloc[:idx + 1].copy()
        temp_df["prediction_text_raw"] = raw_predictions
        temp_df["prediction_text"] = repaired_predictions

        temp_df.to_json(
            f"{output_dir_results}/qwen_lora_full_test_temp_predictions.jsonl",
            orient="records",
            lines=True,
            force_ascii=False
        )

qwen_lora_full_test_df["prediction_text_raw"] = raw_predictions
qwen_lora_full_test_df["prediction_text"] = repaired_predictions

qwen_lora_full_test_df.to_json(
    f"{output_dir_results}/qwen2_5_1_5b_lora_train3000_full_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved full test predictions.")

In [ ]:
qwen_lora_full_metrics = evaluate_structured_predictions(qwen_lora_full_test_df)

qwen_lora_full_metrics = add_equal_weighted_final_scores(qwen_lora_full_metrics)
qwen_lora_full_metrics = add_relation_weighted_final_scores(qwen_lora_full_metrics)
qwen_lora_full_metrics = add_partial_entity_recall(
    qwen_lora_full_metrics,
    qwen_lora_full_test_df
)

qwen_lora_full_metrics

In [ ]:
qwen_lora_full_result = pd.DataFrame([
    {
        "model": "LoRA fine-tuned Qwen2.5-1.5B-Instruct train3000",
        "split": "test",
        **qwen_lora_full_metrics
    }
])

qwen_lora_full_result.to_csv(
    f"{output_dir_results}/qwen2_5_1_5b_lora_train3000_full_test_metrics.csv",
    index=False
)

qwen_lora_full_result

In [ ]:
qwen_lora_full_metrics = {
    "valid_json_rate": 0.968384074941452,
    "exact_match_rate": 0.2611241217798595,
    "entity_micro_precision": 0.8038585209003215,
    "entity_micro_recall": 0.6326825741142444,
    "entity_micro_f1": 0.7080720210398542,
    "entity_example_precision": 0.7840052414408386,
    "entity_example_recall": 0.7052391175049254,
    "entity_example_f1": 0.7289282851035594,
    "relation_micro_precision": 0.4547945205479452,
    "relation_micro_recall": 0.3145925457991156,
    "relation_micro_f1": 0.37191934279312927,
    "relation_example_precision": 0.4760343481654957,
    "relation_example_recall": 0.42101244124077847,
    "relation_example_f1": 0.4344632808987403,
    "final_micro_f1_equal_weighted": 0.5399956819164917,
    "final_example_f1_equal_weighted": 0.5816957830011499,
    "final_micro_f1_relation_weighted": 0.5063804140918193,
    "final_example_f1_relation_weighted": 0.552249282580668,
    "partial_entity_recall": 0.7892657336158507
}

qwen_lora_full_metrics

#Llama-3.2-1B-Instruct

###Llama-3.2-1B-Instruct Zero-shot Direct JSON Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    device_map="auto",
    torch_dtype=torch.float16
)

model.eval()

print("Loaded:", model_name)

In [ ]:
def make_llama_zero_shot_prompt(sentence):
    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific information extraction system. "
                "Extract scientific entities and relations from scientific sentences. "
                "Return only valid JSON. Do not explain anything."
            )
        },
        {
            "role": "user",
            "content": f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method: models, algorithms, architectures, systems, techniques, mechanisms, embeddings, neural networks, classifiers, detectors, or scientific approaches.
- Task: scientific/NLP/CV tasks or problems, such as classification, object detection, machine translation, sentiment analysis, entailment, question answering, or text classification.
- Dataset: dataset or benchmark names, such as MS COCO, ImageNet, BioASQ, SQuAD, MNIST, CIFAR, or other named datasets.

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset as entity types.
- Use only the allowed relation types.
- Do not invent entities.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON.

Sentence:
{sentence}
"""
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
def generate_llama(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
llama_zero_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(llama_zero_test_df.iterrows(), total=len(llama_zero_test_df)):
    prompt = make_llama_zero_shot_prompt(row["sentence"])
    raw_output = generate_llama(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

llama_zero_test_df["prediction_text_raw"] = raw_predictions
llama_zero_test_df["prediction_text"] = repaired_predictions

In [ ]:
llama_zero_test_df["prediction_text"] = llama_zero_test_df["prediction_text"].apply(
    fix_invalid_entity_types
)

In [ ]:
llama_zero_metrics = evaluate_structured_predictions(llama_zero_test_df)

llama_zero_metrics = add_equal_weighted_final_scores(llama_zero_metrics)

llama_zero_metrics = add_relation_weighted_final_scores(
    llama_zero_metrics
)

llama_zero_metrics = add_partial_entity_recall(
    llama_zero_metrics,
    llama_zero_test_df
)

llama_zero_metrics

In [ ]:
llama_zero_result = pd.DataFrame([
    {
        "model": "Zero-shot Llama-3.2-1B-Instruct",
        "split": "test",
        **llama_zero_metrics
    }
])

llama_zero_test_df.to_json(
    f"{output_dir}/llama_3_2_1b_zero_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

llama_zero_result.to_csv(
    f"{output_dir}/llama_3_2_1b_zero_shot_test_metrics.csv",
    index=False
)

llama_zero_result

In [ ]:
llama_zero_metrics = {
    "valid_json_rate": 0.6463700234192038,
    "exact_match_rate": 0.00234192037470726,
    "entity_micro_precision": 0.1633879781420765,
    "entity_micro_recall": 0.10809833694866233,
    "entity_micro_f1": 0.13011314186248912,
    "entity_example_precision": 0.10714750381026727,
    "entity_example_recall": 0.11541164404162063,
    "entity_example_f1": 0.10647802913438677,
    "relation_micro_precision": 0.0,
    "relation_micro_recall": 0.0,
    "relation_micro_f1": 0.0,
    "relation_example_precision": 0.2189695550351288,
    "relation_example_recall": 0.2189695550351288,
    "relation_example_f1": 0.2189695550351288,
    "final_micro_f1_equal_weighted": 0.06505657093124456,
    "final_example_f1_equal_weighted": 0.1627237920847578,
    "final_micro_f1_relation_weighted": 0.052045256744995654,
    "final_example_f1_relation_weighted": 0.173972944674832,
    "partial_entity_recall": 0.21710256631216818
}

llama_zero_metrics

##Llama-3.2-1B-Instruct Few-shot Direct JSON Generation

In [ ]:
def make_llama_few_shot_prompt(sentence, n_examples=3):
    examples_text = []

    for i in range(n_examples):
        example_sentence = train_df.iloc[i]["sentence"]
        example_json = train_df.iloc[i]["target_json"]

        example_block = f"""
Example {i + 1}

Sentence:
{example_sentence}

Output JSON:
{example_json}
"""
        examples_text.append(example_block.strip())

    examples_text = "\n\n".join(examples_text)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a scientific information extraction system. "
                "Extract scientific entities and relations from scientific sentences. "
                "Return only valid JSON. Do not explain anything."
            )
        },
        {
            "role": "user",
            "content": f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method
- Task
- Dataset

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset.
- Use only the allowed relation types.
- Do not invent entities.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON.

Few-shot examples:

{examples_text}

Now extract from this sentence:

Sentence:
{sentence}

Output JSON:
"""
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
llama_few_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(llama_few_test_df.iterrows(), total=len(llama_few_test_df)):
    prompt = make_llama_few_shot_prompt(row["sentence"], n_examples=3)
    raw_output = generate_llama(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

llama_few_test_df["prediction_text_raw"] = raw_predictions
llama_few_test_df["prediction_text"] = repaired_predictions

In [ ]:
llama_few_test_df["prediction_text"] = llama_few_test_df["prediction_text"].apply(
    fix_invalid_entity_types
)

In [ ]:
llama_few_metrics = evaluate_structured_predictions(llama_few_test_df)

llama_few_metrics = add_equal_weighted_final_scores(llama_few_metrics)

llama_few_metrics = add_relation_weighted_final_scores(
    llama_few_metrics
)

llama_few_metrics = add_partial_entity_recall(
    llama_few_metrics,
    llama_few_test_df
)

llama_few_metrics

In [ ]:
llama_few_result = pd.DataFrame([
    {
        "model": "Few-shot Llama-3.2-1B-Instruct",
        "split": "test",
        **llama_few_metrics
    }
])

llama_few_test_df.to_json(
    f"{output_dir}/llama_3_2_1b_few_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

llama_few_result.to_csv(
    f"{output_dir}/llama_3_2_1b_few_shot_test_metrics.csv",
    index=False
)

llama_few_result

In [ ]:
llama_few_metrics = {
    "valid_json_rate": 0.7049180327868853,
    "exact_match_rate": 0.0,
    "entity_micro_precision": 0.3236051502145923,
    "entity_micro_recall": 0.2725958062183659,
    "entity_micro_f1": 0.2959183673469387,
    "entity_example_precision": 0.26830712456881606,
    "entity_example_recall": 0.29248241025173105,
    "entity_example_f1": 0.263643777668908,
    "relation_micro_precision": 0.017227877838684416,
    "relation_micro_recall": 0.013897662665824383,
    "relation_micro_f1": 0.015384615384615384,
    "relation_example_precision": 0.10782591725214676,
    "relation_example_recall": 0.10922921081000705,
    "relation_example_f1": 0.10693755892819125,
    "final_micro_f1_equal_weighted": 0.15565149136577705,
    "final_example_f1_equal_weighted": 0.18529066829854962,
    "final_micro_f1_relation_weighted": 0.12759811616954472,
    "final_example_f1_relation_weighted": 0.16962004642447795,
    "partial_entity_recall": 0.4061371071031492
}

llama_few_metrics

#Llama-3.2-1B-Instruct LoRA Fine-tuning

In [ ]:
!pip install -q transformers==4.51.3 accelerate==1.6.0 bitsandbytes==0.45.5 peft datasets

import os
import json
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

In [ ]:
model_name = "meta-llama/Llama-3.2-1B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    quantization_config=bnb_config,
    device_map={"": 0}
)

model.config.use_cache = False

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj"
    ],
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM"
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = """
You are a scientific information extraction system.
Extract scientific entities and relations from scientific sentences.
Return only valid JSON. Do not explain anything.
""".strip()


def build_user_prompt(sentence):
    return f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method: models, algorithms, architectures, systems, techniques, mechanisms, embeddings, neural networks, classifiers, detectors, or scientific approaches.
- Task: scientific/NLP/CV tasks or problems, such as classification, object detection, machine translation, sentiment analysis, entailment, question answering, or text classification.
- Dataset: dataset or benchmark names, such as MS COCO, ImageNet, BioASQ, SQuAD, MNIST, CIFAR, or other named datasets.

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset as entity types.
- Use only the allowed relation types.
- Do not invent entities.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON.

Sentence:
{sentence}
""".strip()


def build_training_text(sentence, target_json):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(sentence)},
        {"role": "assistant", "content": target_json}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [ ]:
train_ft = train_df.sample(
    n=min(3000, len(train_df)),
    random_state=42
).reset_index(drop=True)

dev_ft = dev_df.sample(
    n=min(300, len(dev_df)),
    random_state=42
).reset_index(drop=True)

test_ft = test_df.copy().reset_index(drop=True)

print("Train fine-tuning size:", len(train_ft))
print("Dev size:", len(dev_ft))
print("Full test size:", len(test_ft))

train_ft["text"] = train_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

dev_ft["text"] = dev_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

train_dataset = Dataset.from_pandas(train_ft[["text"]])
dev_dataset = Dataset.from_pandas(dev_ft[["text"]])

In [ ]:
max_length = 1024

def tokenize_function(batch):
    outputs = tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False
    )

    outputs["labels"] = outputs["input_ids"].copy()
    return outputs


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_dev = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dev_dataset.column_names
)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

In [ ]:
# Smoke training: quick check before full LoRA training

smoke_train_dataset = tokenized_train.select(range(min(16, len(tokenized_train))))
smoke_dev_dataset = tokenized_dev.select(range(min(8, len(tokenized_dev))))

smoke_output_dir = f"{project_dir}/smoke_llama_lora_check"
os.makedirs(smoke_output_dir, exist_ok=True)

smoke_training_args = TrainingArguments(
    output_dir=smoke_output_dir,

    max_steps=5,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=2,

    logging_steps=1,
    eval_strategy="steps",
    eval_steps=5,

    save_strategy="no",
    fp16=True,
    report_to="none",

    optim="paged_adamw_8bit"
)

smoke_trainer = Trainer(
    model=model,
    args=smoke_training_args,
    train_dataset=smoke_train_dataset,
    eval_dataset=smoke_dev_dataset,
    data_collator=data_collator
)

smoke_trainer.train()

print("Smoke training finished successfully.")

In [ ]:
test_sentence = test_df.iloc[0]["sentence"]

raw_output = generate_llama_lora(test_sentence)
repaired_output = repair_prediction_text(raw_output)

print("SENTENCE:")
print(test_sentence)

print("\nRAW OUTPUT:")
print(raw_output)

print("\nREPAIRED OUTPUT:")
print(repaired_output)

In [ ]:
llama_lora_output_dir = f"{project_dir}/llama_3_2_1b_lora_scier_train3000"
os.makedirs(llama_lora_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=llama_lora_output_dir,

    num_train_epochs=2,
    learning_rate=2e-4,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,

    logging_steps=25,
    eval_strategy="steps",
    eval_steps=100,
    save_strategy="steps",
    save_steps=100,

    save_total_limit=2,
    load_best_model_at_end=True,

    fp16=True,
    report_to="none",

    optim="paged_adamw_8bit"
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_dev,
    data_collator=data_collator
)

trainer.train()

model.save_pretrained(llama_lora_output_dir)
tokenizer.save_pretrained(llama_lora_output_dir)

print("LoRA adapter saved to:", llama_lora_output_dir)

In [ ]:
model.eval()

def make_llama_lora_prompt(sentence):
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": build_user_prompt(sentence)}
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_llama_lora(sentence, max_new_tokens=256):
    prompt = make_llama_lora_prompt(sentence)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=1024
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.05,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]
    prediction = tokenizer.decode(generated_ids, skip_special_tokens=True)

    return prediction.strip()


output_dir_results = output_dir
os.makedirs(output_dir_results, exist_ok=True)

llama_lora_full_test_df = test_ft.copy()

raw_predictions = []
repaired_predictions = []

for idx, row in tqdm(llama_lora_full_test_df.iterrows(), total=len(llama_lora_full_test_df)):
    raw_output = generate_llama_lora(row["sentence"])
    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

    if (idx + 1) % 100 == 0:
        temp_df = llama_lora_full_test_df.iloc[:idx + 1].copy()
        temp_df["prediction_text_raw"] = raw_predictions
        temp_df["prediction_text"] = repaired_predictions

        temp_df.to_json(
            f"{output_dir_results}/llama_lora_full_test_temp_predictions.jsonl",
            orient="records",
            lines=True,
            force_ascii=False
        )

llama_lora_full_test_df["prediction_text_raw"] = raw_predictions
llama_lora_full_test_df["prediction_text"] = repaired_predictions

llama_lora_full_test_df.to_json(
    f"{output_dir_results}/llama_3_2_1b_lora_train3000_full_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved full test predictions.")

In [ ]:
llama_lora_metrics = evaluate_structured_predictions(llama_lora_full_test_df)

llama_lora_metrics = add_equal_weighted_final_scores(llama_lora_metrics)
llama_lora_metrics = add_relation_weighted_final_scores(llama_lora_metrics)
llama_lora_metrics = add_partial_entity_recall(
    llama_lora_metrics,
    llama_lora_full_test_df
)

llama_lora_metrics

In [ ]:
llama_lora_result = pd.DataFrame([
    {
        "model": "LoRA fine-tuned Llama-3.2-1B-Instruct train3000",
        "split": "test",
        **llama_lora_metrics
    }
])

llama_lora_result.to_csv(
    f"{output_dir_results}/llama_3_2_1b_lora_train3000_full_test_metrics.csv",
    index=False
)

llama_lora_result

In [ ]:
llama_lora_metrics = {
    "valid_json_rate": 0.955503512880562,
    "exact_match_rate": 0.30679156908665106,
    "entity_micro_precision": 0.8025675077467906,
    "entity_micro_recall": 0.6554591467823572,
    "entity_micro_f1": 0.7215920398009951,
    "entity_example_precision": 0.7728197836511653,
    "entity_example_recall": 0.7318770677669976,
    "entity_example_f1": 0.7420523530593789,
    "relation_micro_precision": 0.4507042253521127,
    "relation_micro_recall": 0.32343651295009473,
    "relation_micro_f1": 0.3766090474439132,
    "relation_example_precision": 0.505743280918925,
    "relation_example_recall": 0.47606058071514745,
    "relation_example_f1": 0.4793486518379754,
    "final_micro_f1_equal_weighted": 0.5491005436224541,
    "final_example_f1_equal_weighted": 0.6107005024486771,
    "final_micro_f1_relation_weighted": 0.514602244386746,
    "final_example_f1_relation_weighted": 0.5844301323265368,
    "partial_entity_recall": 0.8162545072673878
}
llama_lora_metrics

#Gemma-2-2B-it

###Gemma-2-2B-it Zero-shot Direct JSON Generation

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "google/gemma-2-2b-it"

tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    token=hf_token,
    device_map="auto",
    torch_dtype=torch.float16,
    attn_implementation="eager"
)

model.eval()

print("Loaded:", model_name)

In [ ]:
def make_gemma_zero_shot_prompt(sentence):
    messages = [
        {
            "role": "user",
            "content": f"""
You are a scientific information extraction system.
Extract scientific entities and relations from the sentence.
Return only valid JSON. Do not explain anything.

Allowed entity types:
- Method
- Task
- Dataset

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset as entity types.
- Use only the allowed relation types.
- Do not invent entities.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON.

Sentence:
{sentence}
""".strip()
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
def generate_gemma(prompt, max_new_tokens=256):
    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=2048
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.1,
            pad_token_id=tokenizer.eos_token_id,
            eos_token_id=tokenizer.eos_token_id
        )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    prediction = tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    )

    return prediction.strip()

In [ ]:
gemma_zero_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(gemma_zero_test_df.iterrows(), total=len(gemma_zero_test_df)):
    prompt = make_gemma_zero_shot_prompt(row["sentence"])
    raw_output = generate_gemma(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

gemma_zero_test_df["prediction_text_raw"] = raw_predictions
gemma_zero_test_df["prediction_text"] = repaired_predictions

In [ ]:
gemma_zero_test_df["prediction_text"] = gemma_zero_test_df["prediction_text"].apply(
    fix_invalid_entity_types
)

In [ ]:
gemma_zero_metrics = evaluate_structured_predictions(gemma_zero_test_df)

gemma_zero_metrics = add_equal_weighted_final_scores(gemma_zero_metrics)

gemma_zero_metrics = add_relation_weighted_final_scores(
    gemma_zero_metrics
)

gemma_zero_metrics = add_partial_entity_recall(
    gemma_zero_metrics,
    gemma_zero_test_df
)

gemma_zero_metrics

In [ ]:
gemma_zero_result = pd.DataFrame([
    {
        "model": "Zero-shot Gemma-2-2B-it",
        "split": "test",
        **gemma_zero_metrics
    }
])

gemma_zero_test_df.to_json(
    f"{output_dir}/gemma_2_2b_zero_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

gemma_zero_result.to_csv(
    f"{output_dir}/gemma_2_2b_zero_shot_test_metrics.csv",
    index=False
)

gemma_zero_result

In [ ]:
gemma_zero_metrics = {
    "valid_json_rate": 0.9039812646370023,
    "exact_match_rate": 0.00819672131147541,
    "entity_micro_precision": 0.4954337899543379,
    "entity_micro_recall": 0.39226319595083153,
    "entity_micro_f1": 0.4378531073446328,
    "entity_example_precision": 0.4674035566307229,
    "entity_example_recall": 0.42729659424624294,
    "entity_example_f1": 0.42685694477263564,
    "relation_micro_precision": 0.04689564068692206,
    "relation_micro_recall": 0.04485154769425142,
    "relation_micro_f1": 0.045850823377462054,
    "relation_example_precision": 0.07572209211553474,
    "relation_example_recall": 0.06649659863945578,
    "relation_example_f1": 0.06771808077428686,
    "final_micro_f1_equal_weighted": 0.24185196536104742,
    "final_example_f1_equal_weighted": 0.24728751277346125,
    "final_micro_f1_relation_weighted": 0.20265173696433036,
    "final_example_f1_relation_weighted": 0.2113736263736264,
    "partial_entity_recall": 0.6108311282784351
}

gemma_zero_metrics

#Gemma-2-2B-it Few-shot Direct JSON Generation

In [ ]:
def make_gemma_few_shot_prompt(sentence, n_examples=3):
    examples_text = []

    for i in range(n_examples):
        example_sentence = train_df.iloc[i]["sentence"]
        example_json = train_df.iloc[i]["target_json"]

        example_block = f"""
Example {i + 1}

Sentence:
{example_sentence}

Output JSON:
{example_json}
"""
        examples_text.append(example_block.strip())

    examples_text = "\n\n".join(examples_text)

    messages = [
        {
            "role": "user",
            "content": f"""
You are a scientific information extraction system.
Extract scientific entities and relations from scientific sentences.
Return only valid JSON. Do not explain anything.

Allowed entity types:
- Method
- Task
- Dataset

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset.
- Use only the allowed relation types.
- Do not invent entities.
- Every relation head and tail must appear exactly in the entities list.
- If there are no entities, return "entities": [].
- If there are no relations, return "relations": [].
- Return only JSON.

Few-shot examples:

{examples_text}

Now extract from this sentence:

Sentence:
{sentence}

Output JSON:
""".strip()
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

In [ ]:
gemma_few_test_df = test_df.copy()

raw_predictions = []
repaired_predictions = []

for _, row in tqdm(gemma_few_test_df.iterrows(), total=len(gemma_few_test_df)):
    prompt = make_gemma_few_shot_prompt(row["sentence"], n_examples=3)
    raw_output = generate_gemma(prompt)

    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

gemma_few_test_df["prediction_text_raw"] = raw_predictions
gemma_few_test_df["prediction_text"] = repaired_predictions

In [ ]:
gemma_few_test_df["prediction_text"] = gemma_few_test_df["prediction_text"].apply(
    fix_invalid_entity_types
)

In [ ]:
gemma_few_metrics = evaluate_structured_predictions(gemma_few_test_df)

gemma_few_metrics = add_equal_weighted_final_scores(gemma_few_metrics)

gemma_few_metrics = add_relation_weighted_final_scores(
    gemma_few_metrics
)

gemma_few_metrics = add_partial_entity_recall(
    gemma_few_metrics,
    gemma_few_test_df
)

gemma_few_metrics

In [ ]:
gemma_few_result = pd.DataFrame([
    {
        "model": "Few-shot Gemma-2-2B-it",
        "split": "test",
        **gemma_few_metrics
    }
])

gemma_few_test_df.to_json(
    f"{output_dir}/gemma_2_2b_few_shot_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

gemma_few_result.to_csv(
    f"{output_dir}/gemma_2_2b_few_shot_test_metrics.csv",
    index=False
)

gemma_few_result

In [ ]:
gemma_few_metrics = {
    "valid_json_rate": 0.9461358313817331,
    "exact_match_rate": 0.02107728337236534,
    "entity_micro_precision": 0.5401174168297456,
    "entity_micro_recall": 0.49891540130151846,
    "entity_micro_f1": 0.5186994925765833,
    "entity_example_precision": 0.5158691555647059,
    "entity_example_recall": 0.5299726100077389,
    "entity_example_f1": 0.5067064839374712,
    "relation_micro_precision": 0.06824324324324324,
    "relation_micro_recall": 0.06380290587492103,
    "relation_micro_f1": 0.06594841658504734,
    "relation_example_precision": 0.1239935318389651,
    "relation_example_recall": 0.12718879156232787,
    "relation_example_f1": 0.12132700265707567,
    "final_micro_f1_equal_weighted": 0.29232395458081534,
    "final_example_f1_equal_weighted": 0.3140167432972734,
    "final_micro_f1_relation_weighted": 0.24704884698166174,
    "final_example_f1_relation_weighted": 0.2754787951692339,
    "partial_entity_recall": 0.6897361875699113
}

gemma_few_metrics

#Gemma-2-2B-it LoRA Fine-tuning

In [ ]:
import os

os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["WANDB_DISABLED"] = "true"

!pip install -q "bitsandbytes>=0.46.1" transformers==4.51.3 accelerate==1.6.0 peft datasets

import gc
import re
import json
import glob
import torch
import pandas as pd
from tqdm.auto import tqdm
from datasets import Dataset
from torch.utils.data import DataLoader

from huggingface_hub import login

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training,
    PeftModel
)

print("CUDA available:", torch.cuda.is_available())
print("Visible GPU count:", torch.cuda.device_count())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print("Capability:", torch.cuda.get_device_capability(0))

In [ ]:
from huggingface_hub import login

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

print("Logged in to Hugging Face.")

In [ ]:
base_model_name = "google/gemma-2-2b-it"

gemma_lora_output_dir = f"{project_dir}/gemma_2_2b_lora_scier_train5000"
os.makedirs(gemma_lora_output_dir, exist_ok=True)


def find_latest_epoch_checkpoint(output_dir):
    pattern = os.path.join(output_dir, "epoch-*")
    checkpoints = glob.glob(pattern)

    if not checkpoints:
        return None, 0

    def extract_epoch(path):
        name = os.path.basename(path)
        return int(name.split("-")[-1])

    checkpoints = sorted(checkpoints, key=extract_epoch)
    latest = checkpoints[-1]
    latest_epoch = extract_epoch(latest)

    return latest, latest_epoch


latest_checkpoint, start_epoch = find_latest_epoch_checkpoint(gemma_lora_output_dir)

print("Latest checkpoint:", latest_checkpoint)
print("Start epoch:", start_epoch)

In [ ]:
gc.collect()
torch.cuda.empty_cache()

tokenizer_source = latest_checkpoint if latest_checkpoint is not None else base_model_name

tokenizer = AutoTokenizer.from_pretrained(
    tokenizer_source,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    token=hf_token,
    quantization_config=bnb_config,
    device_map={"": 0},
    attn_implementation="eager"
)

base_model.config.use_cache = False

if latest_checkpoint is None:
    base_model = prepare_model_for_kbit_training(base_model)

    lora_config = LoraConfig(
        r=8,
        lora_alpha=16,
        target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj",
            "gate_proj", "up_proj", "down_proj"
        ],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM"
    )

    model = get_peft_model(base_model, lora_config)

else:
    model = PeftModel.from_pretrained(
        base_model,
        latest_checkpoint,
        is_trainable=True
    )

model.print_trainable_parameters()

In [ ]:
SYSTEM_PROMPT = """
You are a scientific information extraction system.
Extract scientific entities and relations from scientific sentences.
Return only valid JSON. Do not explain anything.
""".strip()


def build_user_prompt(sentence):
    return f"""
Extract scientific entities and relations from the sentence.

Allowed entity types:
- Method
- Task
- Dataset

Allowed relation types:
- Part-Of
- Used-For
- Evaluated-With
- Synonym-Of
- Compare-With
- SubClass-Of
- Benchmark-For
- SubTask-Of
- Trained-With

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ],
  "relations": [
    {{"head": "entity text", "relation": "one allowed relation type", "tail": "entity text"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset.
- Use only the allowed relation types.
- Every relation head and tail must appear in the entities list.
- If no entities, return "entities": [].
- If no relations, return "relations": [].
- Return only JSON.

Sentence:
{sentence}
""".strip()


def build_training_text(sentence, target_json):
    messages = [
        {
            "role": "user",
            "content": SYSTEM_PROMPT + "\n\n" + build_user_prompt(sentence)
        },
        {
            "role": "assistant",
            "content": target_json
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=False
    )

In [ ]:
train_size = min(5000, len(train_df))
dev_size = min(713, len(dev_df))

train_ft = train_df.sample(
    n=train_size,
    random_state=42
).reset_index(drop=True)

dev_ft = dev_df.sample(
    n=dev_size,
    random_state=42
).reset_index(drop=True)

test_ft = test_df.copy().reset_index(drop=True)

train_ft["text"] = train_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

dev_ft["text"] = dev_ft.apply(
    lambda row: build_training_text(row["sentence"], row["target_json"]),
    axis=1
)

train_dataset = Dataset.from_pandas(train_ft[["text"]])
dev_dataset = Dataset.from_pandas(dev_ft[["text"]])

print("Train size:", len(train_dataset))
print("Dev size:", len(dev_dataset))
print("Full test size:", len(test_ft))

In [ ]:
max_length = 768

def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        max_length=max_length,
        padding=False
    )


tokenized_train = train_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=train_dataset.column_names
)

tokenized_dev = dev_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=dev_dataset.column_names
)


def custom_causal_lm_collator(features):
    batch = tokenizer.pad(
        features,
        padding=True,
        return_tensors="pt"
    )

    labels = batch["input_ids"].clone()
    labels[batch["attention_mask"] == 0] = -100
    batch["labels"] = labels

    return batch

###Smoke Forward Check

In [ ]:
sample_batch = custom_causal_lm_collator(
    [tokenized_train[i] for i in range(2)]
)

print("input_ids:", sample_batch["input_ids"].shape)
print("attention_mask:", sample_batch["attention_mask"].shape)
print("labels:", sample_batch["labels"].shape)

model.train()

sample_batch_gpu = {
    k: v.to(model.device)
    for k, v in sample_batch.items()
}

with torch.no_grad():
    out = model(**sample_batch_gpu)

print("Smoke forward loss:", out.loss.item())

In [ ]:
batch_size = 1
gradient_accumulation_steps = 8
num_epochs = 2
learning_rate = 2e-4

train_loader = DataLoader(
    tokenized_train,
    batch_size=batch_size,
    shuffle=True,
    collate_fn=custom_causal_lm_collator
)

optimizer = torch.optim.AdamW(
    [p for p in model.parameters() if p.requires_grad],
    lr=learning_rate
)

optimizer_state_path = None

if latest_checkpoint is not None:
    optimizer_state_path = os.path.join(latest_checkpoint, "optimizer.pt")

    if os.path.exists(optimizer_state_path):
        optimizer.load_state_dict(
            torch.load(optimizer_state_path, map_location=model.device)
        )
        print("Loaded optimizer state from:", optimizer_state_path)
    else:
        print("No optimizer state found. Continuing with fresh optimizer.")

model.train()

for epoch in range(start_epoch, num_epochs):
    print(f"\nStarting epoch {epoch + 1}/{num_epochs}")

    running_loss = 0.0
    optimizer.zero_grad()

    for step, batch in enumerate(tqdm(train_loader)):
        batch = {
            k: v.to(model.device)
            for k, v in batch.items()
        }

        outputs = model(**batch)
        loss = outputs.loss / gradient_accumulation_steps

        loss.backward()

        running_loss += loss.item()

        if (step + 1) % gradient_accumulation_steps == 0:
            optimizer.step()
            optimizer.zero_grad()

        if (step + 1) % 200 == 0:
            avg_loss = running_loss / 200
            print(
                f"epoch={epoch + 1}, step={step + 1}, "
                f"avg_scaled_loss={avg_loss:.4f}"
            )
            running_loss = 0.0

    epoch_dir = os.path.join(gemma_lora_output_dir, f"epoch-{epoch + 1}")
    os.makedirs(epoch_dir, exist_ok=True)

    model.save_pretrained(epoch_dir)
    tokenizer.save_pretrained(epoch_dir)

    torch.save(
        optimizer.state_dict(),
        os.path.join(epoch_dir, "optimizer.pt")
    )

    with open(os.path.join(epoch_dir, "training_state.json"), "w") as f:
        json.dump(
            {
                "completed_epoch": epoch + 1,
                "num_epochs": num_epochs,
                "train_size": train_size,
                "max_length": max_length,
                "learning_rate": learning_rate,
                "gradient_accumulation_steps": gradient_accumulation_steps
            },
            f,
            indent=2
        )

    print("Saved epoch checkpoint:", epoch_dir)

model.save_pretrained(gemma_lora_output_dir)
tokenizer.save_pretrained(gemma_lora_output_dir)

print("Final adapter saved to:", gemma_lora_output_dir)

###Inference Functions

In [ ]:
model.eval()

def extract_json_substring(text):
    text = str(text).strip()

    start = text.find("{")
    end = text.rfind("}")

    if start != -1 and end != -1 and end > start:
        return text[start:end + 1]

    return text


def repair_prediction_text(text):
    candidate = extract_json_substring(text)

    try:
        obj = json.loads(candidate)
        return json.dumps(obj, ensure_ascii=False, sort_keys=True)
    except Exception:
        return candidate


def make_gemma_lora_prompt(sentence):
    messages = [
        {
            "role": "user",
            "content": SYSTEM_PROMPT + "\n\n" + build_user_prompt(sentence)
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def generate_gemma_lora(sentence, max_new_tokens=256):
    prompt = make_gemma_lora_prompt(sentence)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=768
    ).to(model.device)

    with torch.no_grad():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.05,
                pad_token_id=tokenizer.eos_token_id,
                use_cache=False
            )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

In [ ]:
output_dir_results = output_dir
os.makedirs(output_dir_results, exist_ok=True)

gemma_lora_full_test_df = test_df.copy().reset_index(drop=True)

raw_predictions = []
repaired_predictions = []

for idx, row in tqdm(gemma_lora_full_test_df.iterrows(), total=len(gemma_lora_full_test_df)):
    raw_output = generate_gemma_lora(row["sentence"])
    repaired_output = repair_prediction_text(raw_output)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

    if (idx + 1) % 100 == 0:
        temp_df = gemma_lora_full_test_df.iloc[:idx + 1].copy()
        temp_df["prediction_text_raw"] = raw_predictions
        temp_df["prediction_text"] = repaired_predictions

        temp_df.to_json(
            f"{output_dir_results}/gemma_lora_full_test_temp_predictions.jsonl",
            orient="records",
            lines=True,
            force_ascii=False
        )

gemma_lora_full_test_df["prediction_text_raw"] = raw_predictions
gemma_lora_full_test_df["prediction_text"] = repaired_predictions

gemma_lora_full_test_df.to_json(
    f"{output_dir_results}/gemma_2_2b_lora_train5000_full_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved Gemma LoRA full test predictions.")

In [ ]:
gemma_lora_metrics = evaluate_structured_predictions(gemma_lora_full_test_df)

gemma_lora_metrics = add_equal_weighted_final_scores(gemma_lora_metrics)
gemma_lora_metrics = add_relation_weighted_final_scores(gemma_lora_metrics)
gemma_lora_metrics = add_partial_entity_recall(
    gemma_lora_metrics,
    gemma_lora_full_test_df
)

gemma_lora_metrics

In [ ]:
gemma_lora_result = pd.DataFrame([
    {
        "model": "LoRA fine-tuned Gemma-2-2B-it train5000",
        "split": "test",
        **gemma_lora_metrics
    }
])

gemma_lora_result.to_csv(
    f"{output_dir_results}/gemma_2_2b_lora_train5000_full_test_metrics.csv",
    index=False
)

gemma_lora_result

In [ ]:
gemma_lora_metrics = {
    "valid_json_rate": 0.9449648711943794,
    "exact_match_rate": 0.2728337236533958,
    "entity_micro_precision": 0.7819672131147541,
    "entity_micro_recall": 0.6898047722342733,
    "entity_micro_f1": 0.7330003841721091,
    "entity_example_precision": 0.7592547674807628,
    "entity_example_recall": 0.7657243225158916,
    "entity_example_f1": 0.7528577893383652,
    "relation_micro_precision": 0.4551907719609583,
    "relation_micro_recall": 0.3240682248894504,
    "relation_micro_f1": 0.3785977859778598,
    "relation_example_precision": 0.4783372365339578,
    "relation_example_recall": 0.4786662205865953,
    "relation_example_f1": 0.4677244821272924,
    "final_micro_f1_equal_weighted": 0.5557990850749844,
    "final_example_f1_equal_weighted": 0.6102911357328288,
    "final_micro_f1_relation_weighted": 0.5203588252555595,
    "final_example_f1_relation_weighted": 0.5817778050117215,
    "partial_entity_recall": 0.8507244999341018
}

gemma_lora_metrics

#SciBERT Relation Classifier with Gold Entities

###SciBERT predicts relations between gold entity pairs. This is an upper-bound/diagnostic relation extraction experiment.

In [ ]:
!pip install -q transformers==4.51.3 datasets==3.5.1 accelerate==1.6.0 scikit-learn

import os
import json
import random
import numpy as np
import pandas as pd
import torch

from tqdm.auto import tqdm
from datasets import Dataset

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

from sklearn.metrics import f1_score, classification_report

print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

In [ ]:
RELATION_TYPES = [
    "No-Relation",
    "Part-Of",
    "Used-For",
    "Evaluated-With",
    "Synonym-Of",
    "Compare-With",
    "SubClass-Of",
    "Benchmark-For",
    "SubTask-Of",
    "Trained-With"
]

label2id = {label: i for i, label in enumerate(RELATION_TYPES)}
id2label = {i: label for label, i in label2id.items()}

ENTITY_TYPES = ["Method", "Task", "Dataset"]

print(label2id)

In [ ]:
def get_entities_and_relations(row):
    obj, _ = parse_json_safe(row["target_json"])

    entities = obj.get("entities", [])
    relations = obj.get("relations", [])

    clean_entities = []
    seen_entities = set()

    for ent in entities:
        if not isinstance(ent, dict):
            continue

        text = str(ent.get("text", "")).strip()
        ent_type = str(ent.get("type", "")).strip()

        if not text or ent_type not in ENTITY_TYPES:
            continue

        key = (text, ent_type)

        if key not in seen_entities:
            clean_entities.append({
                "text": text,
                "type": ent_type
            })
            seen_entities.add(key)

    clean_relations = []

    for rel in relations:
        if not isinstance(rel, dict):
            continue

        head = str(rel.get("head", "")).strip()
        relation = str(rel.get("relation", "")).strip()
        tail = str(rel.get("tail", "")).strip()

        if head and tail and relation in RELATION_TYPES:
            clean_relations.append({
                "head": head,
                "relation": relation,
                "tail": tail
            })

    return clean_entities, clean_relations

In [ ]:
def mark_entity_pair_in_sentence(sentence, head_text, head_type, tail_text, tail_type):
    sentence = str(sentence)

    head_start = sentence.find(head_text)
    tail_start = sentence.find(tail_text)

    if head_start == -1 or tail_start == -1:
        return None

    head_end = head_start + len(head_text)
    tail_end = tail_start + len(tail_text)

    if not (head_end <= tail_start or tail_end <= head_start):
        return None

    spans = [
        {
            "start": head_start,
            "end": head_end,
            "start_marker": f"[HEAD_{head_type}]",
            "end_marker": f"[/HEAD_{head_type}]"
        },
        {
            "start": tail_start,
            "end": tail_end,
            "start_marker": f"[TAIL_{tail_type}]",
            "end_marker": f"[/TAIL_{tail_type}]"
        }
    ]

    spans = sorted(spans, key=lambda x: x["start"], reverse=True)
    marked_sentence = sentence

    for span in spans:
        marked_sentence = (
            marked_sentence[:span["start"]]
            + span["start_marker"]
            + " "
            + marked_sentence[span["start"]:span["end"]]
            + " "
            + span["end_marker"]
            + marked_sentence[span["end"]:]
        )

    return marked_sentence


def make_pair_text(sentence, head, head_type, tail, tail_type):
    marked_sentence = mark_entity_pair_in_sentence(
        sentence,
        head,
        head_type,
        tail,
        tail_type
    )

    if marked_sentence is not None:
        return (
            f"Sentence: {marked_sentence}\n"
            f"Head entity type: {head_type}\n"
            f"Tail entity type: {tail_type}\n"
            f"Classify the directed relation from HEAD to TAIL."
        )

    return (
        f"Sentence: {sentence}\n"
        f"Head entity: {head} [type: {head_type}]\n"
        f"Tail entity: {tail} [type: {tail_type}]\n"
        f"Classify the directed relation from head entity to tail entity."
    )

In [ ]:
def build_relation_examples_from_df(
    df,
    max_negative_per_positive=3,
    max_negative_if_no_positive=3,
    random_state=42
):
    rng = random.Random(random_state)
    examples = []

    for _, row in tqdm(df.iterrows(), total=len(df)):
        sentence = row["sentence"]
        entities, relations = get_entities_and_relations(row)

        rel_lookup = {}

        for rel in relations:
            rel_lookup[(rel["head"], rel["tail"])] = rel["relation"]

        positive_examples = []
        negative_examples = []

        for i, head_ent in enumerate(entities):
            for j, tail_ent in enumerate(entities):
                if i == j:
                    continue

                head = head_ent["text"]
                tail = tail_ent["text"]

                label_name = rel_lookup.get((head, tail), "No-Relation")

                example = {
                    "text": make_pair_text(
                        sentence,
                        head,
                        head_ent["type"],
                        tail,
                        tail_ent["type"]
                    ),
                    "label": label2id[label_name],
                    "label_name": label_name
                }

                if label_name == "No-Relation":
                    negative_examples.append(example)
                else:
                    positive_examples.append(example)

        examples.extend(positive_examples)

        if len(positive_examples) > 0:
            max_negatives = max_negative_per_positive * len(positive_examples)

            if len(negative_examples) > max_negatives:
                negative_examples = rng.sample(negative_examples, max_negatives)

            examples.extend(negative_examples)

        else:
            if len(negative_examples) > max_negative_if_no_positive:
                negative_examples = rng.sample(
                    negative_examples,
                    max_negative_if_no_positive
                )

            examples.extend(negative_examples)

    return pd.DataFrame(examples)

In [ ]:
train_rel_df = build_relation_examples_from_df(
    train_df,
    max_negative_per_positive=3,
    max_negative_if_no_positive=3,
    random_state=42
)

dev_rel_df = build_relation_examples_from_df(
    dev_df,
    max_negative_per_positive=3,
    max_negative_if_no_positive=3,
    random_state=43
)

print("Train relation examples:", train_rel_df.shape)
print("Dev relation examples:", dev_rel_df.shape)

print("\nTrain label distribution:")
print(train_rel_df["label_name"].value_counts())

train_rel_df.head()

In [ ]:
relation_model_name = "allenai/scibert_scivocab_uncased"

rel_tokenizer = AutoTokenizer.from_pretrained(relation_model_name)

rel_model = AutoModelForSequenceClassification.from_pretrained(
    relation_model_name,
    num_labels=len(RELATION_TYPES),
    id2label=id2label,
    label2id=label2id
)

special_tokens = []

for ent_type in ENTITY_TYPES:
    special_tokens.extend([
        f"[HEAD_{ent_type}]",
        f"[/HEAD_{ent_type}]",
        f"[TAIL_{ent_type}]",
        f"[/TAIL_{ent_type}]"
    ])

num_added = rel_tokenizer.add_special_tokens({
    "additional_special_tokens": special_tokens
})

rel_model.resize_token_embeddings(len(rel_tokenizer))

print("Loaded:", relation_model_name)
print("Added special tokens:", num_added)
print("Tokenizer size:", len(rel_tokenizer))

In [ ]:
train_rel_dataset = Dataset.from_pandas(train_rel_df[["text", "label"]])
dev_rel_dataset = Dataset.from_pandas(dev_rel_df[["text", "label"]])


def tokenize_relation_batch(batch):
    return rel_tokenizer(
        batch["text"],
        truncation=True,
        max_length=256
    )


tokenized_train_rel = train_rel_dataset.map(
    tokenize_relation_batch,
    batched=True,
    remove_columns=["text"]
)

tokenized_dev_rel = dev_rel_dataset.map(
    tokenize_relation_batch,
    batched=True,
    remove_columns=["text"]
)

rel_collator = DataCollatorWithPadding(tokenizer=rel_tokenizer)

In [ ]:
def compute_relation_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)

    micro_f1 = f1_score(labels, preds, average="micro")
    macro_f1 = f1_score(labels, preds, average="macro")

    relation_label_ids = [
        label2id[label]
        for label in RELATION_TYPES
        if label != "No-Relation"
    ]

    relation_macro_f1 = f1_score(
        labels,
        preds,
        labels=relation_label_ids,
        average="macro",
        zero_division=0
    )

    return {
        "micro_f1": micro_f1,
        "macro_f1": macro_f1,
        "relation_macro_f1_excl_no_relation": relation_macro_f1
    }


rel_output_dir = f"{project_dir}/scibert_relation_classifier_marked"
os.makedirs(rel_output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=rel_output_dir,

    num_train_epochs=4,
    learning_rate=2e-5,

    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    eval_strategy="epoch",
    save_strategy="epoch",

    load_best_model_at_end=True,
    metric_for_best_model="relation_macro_f1_excl_no_relation",
    greater_is_better=True,

    logging_steps=50,
    save_total_limit=2,

    report_to="none"
)

rel_trainer = Trainer(
    model=rel_model,
    args=training_args,
    train_dataset=tokenized_train_rel,
    eval_dataset=tokenized_dev_rel,
    tokenizer=rel_tokenizer,
    data_collator=rel_collator,
    compute_metrics=compute_relation_metrics
)

rel_trainer.train()

rel_trainer.save_model(rel_output_dir)
rel_tokenizer.save_pretrained(rel_output_dir)

print("Saved SciBERT relation classifier to:", rel_output_dir)

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

rel_model.to(device)
rel_model.eval()


def predict_relation_for_pair(sentence, head_ent, tail_ent):
    pair_text = make_pair_text(
        sentence,
        head_ent["text"],
        head_ent["type"],
        tail_ent["text"],
        tail_ent["type"]
    )

    inputs = rel_tokenizer(
        pair_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = rel_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id])

    return pred_label, confidence


def predict_relations_for_sentence(sentence, entities, threshold=0.50):
    relations = []
    seen = set()

    for i, head_ent in enumerate(entities):
        for j, tail_ent in enumerate(entities):
            if i == j:
                continue

            pred_label, confidence = predict_relation_for_pair(
                sentence,
                head_ent,
                tail_ent
            )

            if pred_label == "No-Relation":
                continue

            if confidence < threshold:
                continue

            key = (
                head_ent["text"],
                pred_label,
                tail_ent["text"]
            )

            if key not in seen:
                relations.append({
                    "head": head_ent["text"],
                    "relation": pred_label,
                    "tail": tail_ent["text"]
                })
                seen.add(key)

    return relations

In [ ]:
def build_gold_entity_scibert_json(row, threshold=0.50):
    sentence = row["sentence"]
    entities, _ = get_entities_and_relations(row)

    relations = predict_relations_for_sentence(
        sentence,
        entities,
        threshold=threshold
    )

    final_obj = {
        "entities": entities,
        "relations": relations
    }

    return json.dumps(
        final_obj,
        ensure_ascii=False,
        sort_keys=True
    )

In [ ]:
best_threshold = 0.50

gold_scibert_df = test_df.copy()
gold_scibert_predictions = []

for _, row in tqdm(gold_scibert_df.iterrows(), total=len(gold_scibert_df)):
    pred_text = build_gold_entity_scibert_json(
        row,
        threshold=best_threshold
    )

    gold_scibert_predictions.append(pred_text)

gold_scibert_df["prediction_text"] = gold_scibert_predictions

gold_scibert_metrics = evaluate_structured_predictions(gold_scibert_df)

gold_scibert_metrics = add_equal_weighted_final_scores(gold_scibert_metrics)
gold_scibert_metrics = add_relation_weighted_final_scores(gold_scibert_metrics)
gold_scibert_metrics = add_partial_entity_recall(
    gold_scibert_metrics,
    gold_scibert_df
)

gold_scibert_metrics

In [ ]:
gold_scibert_result = pd.DataFrame([
    {
        "model": "Upper-bound: Gold entities + SciBERT relation classifier with entity markers",
        "split": "test",
        **gold_scibert_metrics
    }
])

gold_scibert_df.to_json(
    f"{output_dir}/upper_bound_gold_entities_scibert_marked_relations_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

gold_scibert_result.to_csv(
    f"{output_dir}/upper_bound_gold_entities_scibert_marked_relations_test_metrics.csv",
    index=False
)

gold_scibert_result

In [ ]:
gold_scibert_metrics = {
    "valid_json_rate": 1.0,
    "exact_match_rate": 0.594847775175644,
    "entity_micro_precision": 1.0,
    "entity_micro_recall": 1.0,
    "entity_micro_f1": 1.0,
    "entity_example_precision": 1.0,
    "entity_example_recall": 1.0,
    "entity_example_f1": 1.0,
    "relation_micro_precision": 0.7619986850756082,
    "relation_micro_recall": 0.7321541377132028,
    "relation_micro_f1": 0.746778350515464,
    "relation_example_precision": 0.75993405826544,
    "relation_example_recall": 0.758122660581677,
    "relation_example_f1": 0.7461446166149949,
    "final_micro_f1_equal_weighted": 0.8733891752577321,
    "final_example_f1_equal_weighted": 0.8730723083074974,
    "final_micro_f1_relation_weighted": 0.8480670103092784,
    "final_example_f1_relation_weighted": 0.847686769968997,
    "partial_entity_recall": 1.0
}

gold_scibert_metrics

#Split-task: Gemma LoRA Entity Extraction + SciBERT Relation Classification

###Gemma predicts entities only; SciBERT predicts relations between predicted entities.

In [ ]:
import gc
import torch

from huggingface_hub import login
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

hf_token = os.environ.get("HF_TOKEN")
login(token=hf_token)

base_model_name = "google/gemma-2-2b-it"

# Change this if your saved adapter is in a different Drive folder
adapter_path = f"{project_dir}/gemma_2_2b_lora_scier_train5000/epoch-2"

gc.collect()
torch.cuda.empty_cache()

tokenizer = AutoTokenizer.from_pretrained(
    adapter_path,
    token=hf_token
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    token=hf_token,
    torch_dtype=torch.float16,
    device_map={"": 0},
    attn_implementation="eager"
)

base_model.config.use_cache = True

gemma_entity_model = PeftModel.from_pretrained(
    base_model,
    adapter_path,
    is_trainable=False
)

gemma_entity_model.eval()

print("Loaded Gemma LoRA adapter for entity extraction.")
print("Adapter path:", adapter_path)

In [ ]:
import re
import json

ENTITY_TYPES = ["Method", "Task", "Dataset"]

ENTITY_SYSTEM_PROMPT = """
You are a scientific entity extraction system.
Extract only scientific entities from the sentence.
Return only valid JSON. Do not explain anything.
""".strip()


def make_entity_only_prompt(sentence):
    messages = [
        {
            "role": "user",
            "content": f"""
{ENTITY_SYSTEM_PROMPT}

Allowed entity types:
- Method
- Task
- Dataset

Return exactly this JSON structure:
{{
  "entities": [
    {{"text": "exact entity span from sentence", "type": "Method or Task or Dataset"}}
  ]
}}

Rules:
- Copy entity text exactly from the sentence.
- Use only Method, Task, Dataset.
- Do not extract relations.
- If no entities, return {{"entities": []}}.
- Return only JSON.

Sentence:
{sentence}
""".strip()
        }
    ]

    return tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )


def repair_entity_prediction(text, sentence=None):
    text = str(text)

    clean_entities = []
    seen = set()

    pattern_1 = r'"text"\s*:\s*"([^"]+)"\s*,\s*"type"\s*:\s*"([^"]+)"'
    pattern_2 = r'"type"\s*:\s*"([^"]+)"\s*,\s*"text"\s*:\s*"([^"]+)"'

    matches = []

    for ent_text, ent_type in re.findall(pattern_1, text):
        matches.append((ent_text, ent_type))

    for ent_type, ent_text in re.findall(pattern_2, text):
        matches.append((ent_text, ent_type))

    for ent_text, ent_type in matches:
        ent_text = str(ent_text).strip()
        ent_type = str(ent_type).strip()

        if not ent_text:
            continue

        if ent_type not in ENTITY_TYPES:
            continue

        if sentence is not None and ent_text not in sentence:
            continue

        key = (ent_text, ent_type)

        if key not in seen:
            clean_entities.append({
                "text": ent_text,
                "type": ent_type
            })
            seen.add(key)

    return json.dumps(
        {"entities": clean_entities},
        ensure_ascii=False,
        sort_keys=True
    )


def generate_gemma_entities(sentence, max_new_tokens=96):
    prompt = make_entity_only_prompt(sentence)

    inputs = tokenizer(
        prompt,
        return_tensors="pt",
        truncation=True,
        max_length=512
    ).to(gemma_entity_model.device)

    with torch.inference_mode():
        with torch.amp.autocast("cuda", dtype=torch.float16):
            outputs = gemma_entity_model.generate(
                **inputs,
                max_new_tokens=max_new_tokens,
                do_sample=False,
                repetition_penalty=1.15,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id,
                eos_token_id=tokenizer.eos_token_id,
                use_cache=True
            )

    generated_ids = outputs[0][inputs["input_ids"].shape[-1]:]

    return tokenizer.decode(
        generated_ids,
        skip_special_tokens=True
    ).strip()

In [ ]:
entity_temp_path = f"{output_dir}/gemma_lora_entity_only_temp_predictions.jsonl"
entity_final_path = f"{output_dir}/gemma_lora_entity_only_test_predictions.jsonl"

gemma_entity_test_df = test_df.copy().reset_index(drop=True)

if os.path.exists(entity_temp_path):
    temp_df = pd.read_json(entity_temp_path, orient="records", lines=True)
    start_idx = len(temp_df)

    raw_predictions = temp_df["entity_prediction_raw"].tolist()
    repaired_predictions = temp_df["entity_prediction_text"].tolist()

    print("Resuming from:", start_idx)
else:
    start_idx = 0
    raw_predictions = []
    repaired_predictions = []

    print("Starting from beginning")

for idx in tqdm(range(start_idx, len(gemma_entity_test_df))):
    sentence = gemma_entity_test_df.iloc[idx]["sentence"]

    raw_output = generate_gemma_entities(sentence, max_new_tokens=96)
    repaired_output = repair_entity_prediction(raw_output, sentence=sentence)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

    if (idx + 1) % 20 == 0:
        temp_df = gemma_entity_test_df.iloc[:idx + 1].copy()
        temp_df["entity_prediction_raw"] = raw_predictions
        temp_df["entity_prediction_text"] = repaired_predictions

        temp_df.to_json(
            entity_temp_path,
            orient="records",
            lines=True,
            force_ascii=False
        )

        print(f"Saved entity progress at {idx + 1}/{len(gemma_entity_test_df)}")

gemma_entity_test_df["entity_prediction_raw"] = raw_predictions
gemma_entity_test_df["entity_prediction_text"] = repaired_predictions

gemma_entity_test_df.to_json(
    entity_final_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved entity-only predictions:", entity_final_path)

In [ ]:
entity_only_eval_df = gemma_entity_test_df.copy()

full_json_predictions = []

for _, row in entity_only_eval_df.iterrows():
    obj, _ = parse_json_safe(row["entity_prediction_text"])

    final_obj = {
        "entities": obj.get("entities", []),
        "relations": []
    }

    full_json_predictions.append(
        json.dumps(final_obj, ensure_ascii=False, sort_keys=True)
    )

entity_only_eval_df["prediction_text"] = full_json_predictions

entity_only_metrics = evaluate_structured_predictions(entity_only_eval_df)
entity_only_metrics = add_equal_weighted_final_scores(entity_only_metrics)
entity_only_metrics = add_relation_weighted_final_scores(entity_only_metrics)
entity_only_metrics = add_partial_entity_recall(
    entity_only_metrics,
    entity_only_eval_df
)

entity_only_metrics

In [ ]:
entity_only_metrics = {
    "valid_json_rate": 1.0,
    "exact_match_rate": 0.12295081967213115,
    "entity_micro_precision": 0.8000758725341427,
    "entity_micro_recall": 0.762472885032538,
    "entity_micro_f1": 0.7808219178082193,
    "entity_example_precision": 0.812989294078287,
    "entity_example_recall": 0.8022742447508255,
    "entity_example_f1": 0.7956290030523807,
    "relation_micro_precision": 0.0,
    "relation_micro_recall": 0.0,
    "relation_micro_f1": 0.0,
    "relation_example_precision": 0.22833723653395785,
    "relation_example_recall": 0.22833723653395785,
    "relation_example_f1": 0.22833723653395785,
    "final_micro_f1_equal_weighted": 0.39041095890410965,
    "final_example_f1_equal_weighted": 0.5119831197931692,
    "final_micro_f1_relation_weighted": 0.3123287671232877,
    "final_example_f1_relation_weighted": 0.455253943141327,
    "partial_entity_recall": 0.8834985824739923
}

entity_only_metrics

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import numpy as np

rel_model_path = f"{project_dir}/scibert_relation_classifier_marked"

rel_tokenizer = AutoTokenizer.from_pretrained(rel_model_path)
rel_model = AutoModelForSequenceClassification.from_pretrained(rel_model_path)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
rel_model.to(device)
rel_model.eval()

print("Loaded SciBERT relation classifier.")
print("Device:", device)
print("Labels:", rel_model.config.id2label)

In [ ]:
RELATION_TYPES = [
    "No-Relation",
    "Part-Of",
    "Used-For",
    "Evaluated-With",
    "Synonym-Of",
    "Compare-With",
    "SubClass-Of",
    "Benchmark-For",
    "SubTask-Of",
    "Trained-With"
]

label2id = {label: i for i, label in enumerate(RELATION_TYPES)}
id2label = {i: label for label, i in label2id.items()}


def get_gemma_predicted_entities(row):
    obj, _ = parse_json_safe(row["entity_prediction_text"])
    entities = obj.get("entities", [])

    clean_entities = []
    seen = set()

    for ent in entities:
        if not isinstance(ent, dict):
            continue

        text = str(ent.get("text", "")).strip()
        ent_type = str(ent.get("type", "")).strip()

        if not text or ent_type not in ENTITY_TYPES:
            continue

        key = (text, ent_type)

        if key not in seen:
            clean_entities.append({
                "text": text,
                "type": ent_type
            })
            seen.add(key)

    return clean_entities


def mark_entity_pair_in_sentence(sentence, head_text, head_type, tail_text, tail_type):
    sentence = str(sentence)

    head_start = sentence.find(head_text)
    tail_start = sentence.find(tail_text)

    if head_start == -1 or tail_start == -1:
        return None

    head_end = head_start + len(head_text)
    tail_end = tail_start + len(tail_text)

    if not (head_end <= tail_start or tail_end <= head_start):
        return None

    spans = [
        {
            "start": head_start,
            "end": head_end,
            "start_marker": f"[HEAD_{head_type}]",
            "end_marker": f"[/HEAD_{head_type}]"
        },
        {
            "start": tail_start,
            "end": tail_end,
            "start_marker": f"[TAIL_{tail_type}]",
            "end_marker": f"[/TAIL_{tail_type}]"
        }
    ]

    spans = sorted(spans, key=lambda x: x["start"], reverse=True)
    marked_sentence = sentence

    for span in spans:
        marked_sentence = (
            marked_sentence[:span["start"]]
            + span["start_marker"]
            + " "
            + marked_sentence[span["start"]:span["end"]]
            + " "
            + span["end_marker"]
            + marked_sentence[span["end"]:]
        )

    return marked_sentence


def make_pair_text(sentence, head, head_type, tail, tail_type):
    marked_sentence = mark_entity_pair_in_sentence(
        sentence,
        head,
        head_type,
        tail,
        tail_type
    )

    if marked_sentence is not None:
        return (
            f"Sentence: {marked_sentence}\n"
            f"Head entity type: {head_type}\n"
            f"Tail entity type: {tail_type}\n"
            f"Classify the directed relation from HEAD to TAIL."
        )

    return (
        f"Sentence: {sentence}\n"
        f"Head entity: {head} [type: {head_type}]\n"
        f"Tail entity: {tail} [type: {tail_type}]\n"
        f"Classify the directed relation from head entity to tail entity."
    )


def predict_relation_for_pair(sentence, head_ent, tail_ent):
    pair_text = make_pair_text(
        sentence,
        head_ent["text"],
        head_ent["type"],
        tail_ent["text"],
        tail_ent["type"]
    )

    inputs = rel_tokenizer(
        pair_text,
        return_tensors="pt",
        truncation=True,
        max_length=256
    ).to(device)

    with torch.no_grad():
        outputs = rel_model(**inputs)

    probs = torch.softmax(outputs.logits, dim=-1)[0].detach().cpu().numpy()

    pred_id = int(np.argmax(probs))
    pred_label = id2label[pred_id]
    confidence = float(probs[pred_id])

    return pred_label, confidence


def predict_relations_for_sentence(sentence, entities, threshold=0.50):
    relations = []
    seen = set()

    for i, head_ent in enumerate(entities):
        for j, tail_ent in enumerate(entities):
            if i == j:
                continue

            pred_label, confidence = predict_relation_for_pair(
                sentence,
                head_ent,
                tail_ent
            )

            if pred_label == "No-Relation":
                continue

            if confidence < threshold:
                continue

            key = (
                head_ent["text"],
                pred_label,
                tail_ent["text"]
            )

            if key not in seen:
                relations.append({
                    "head": head_ent["text"],
                    "relation": pred_label,
                    "tail": tail_ent["text"]
                })
                seen.add(key)

    return relations

In [ ]:
def build_split_task_json_from_gemma(row, threshold=0.50):
    sentence = row["sentence"]

    entities = get_gemma_predicted_entities(row)

    relations = predict_relations_for_sentence(
        sentence,
        entities,
        threshold=threshold
    )

    final_obj = {
        "entities": entities,
        "relations": relations
    }

    return json.dumps(
        final_obj,
        ensure_ascii=False,
        sort_keys=True
    )

In [ ]:
threshold_results = []

for threshold in [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]:
    print("\nRunning threshold:", threshold)

    temp_df = gemma_entity_test_df.copy()
    predictions = []

    for _, row in tqdm(temp_df.iterrows(), total=len(temp_df)):
        pred_text = build_split_task_json_from_gemma(
            row,
            threshold=threshold
        )
        predictions.append(pred_text)

    temp_df["prediction_text"] = predictions

    metrics = evaluate_structured_predictions(temp_df)
    metrics = add_equal_weighted_final_scores(metrics)
    metrics = add_relation_weighted_final_scores(metrics)
    metrics = add_partial_entity_recall(metrics, temp_df)

    threshold_results.append({
        "threshold": threshold,
        **metrics
    })

threshold_results_df = pd.DataFrame(threshold_results)

threshold_results_df[
    [
        "threshold",
        "valid_json_rate",
        "exact_match_rate",
        "entity_micro_f1",
        "relation_micro_precision",
        "relation_micro_recall",
        "relation_micro_f1",
        "final_micro_f1_equal_weighted",
        "partial_entity_recall"
    ]
]

In [ ]:
best_threshold = threshold_results_df.sort_values(
    by="final_micro_f1_equal_weighted",
    ascending=False
).iloc[0]["threshold"]

print("Best threshold:", best_threshold)

split_task_gemma_df = gemma_entity_test_df.copy()
split_predictions = []

for _, row in tqdm(split_task_gemma_df.iterrows(), total=len(split_task_gemma_df)):
    pred_text = build_split_task_json_from_gemma(
        row,
        threshold=best_threshold
    )
    split_predictions.append(pred_text)

split_task_gemma_df["prediction_text"] = split_predictions

split_task_gemma_metrics = evaluate_structured_predictions(split_task_gemma_df)
split_task_gemma_metrics = add_equal_weighted_final_scores(split_task_gemma_metrics)
split_task_gemma_metrics = add_relation_weighted_final_scores(split_task_gemma_metrics)
split_task_gemma_metrics = add_partial_entity_recall(
    split_task_gemma_metrics,
    split_task_gemma_df
)

split_task_gemma_metrics

In [ ]:
split_task_gemma_metrics = {
    "valid_json_rate": 1.0,
    "exact_match_rate": 0.32669789227166274,
    "entity_micro_precision": 0.8000758725341427,
    "entity_micro_recall": 0.762472885032538,
    "entity_micro_f1": 0.7808219178082193,
    "entity_example_precision": 0.812989294078287,
    "entity_example_recall": 0.8022742447508255,
    "entity_example_f1": 0.7956290030523807,
    "relation_micro_precision": 0.5075342465753425,
    "relation_micro_recall": 0.4680985470625395,
    "relation_micro_f1": 0.4870193887610911,
    "relation_example_precision": 0.5762427723200557,
    "relation_example_recall": 0.5702475835379817,
    "relation_example_f1": 0.5569834943336409,
    "final_micro_f1_equal_weighted": 0.6339206532846552,
    "final_example_f1_equal_weighted": 0.6763062486930108,
    "final_micro_f1_relation_weighted": 0.6045404003799424,
    "final_example_f1_relation_weighted": 0.6524416978211369,
    "partial_entity_recall": 0.8834985824739923
}

split_task_gemma_metrics

In [ ]:
split_task_gemma_df.to_json(
    f"{output_dir}/split_task_gemma_entities_scibert_relations_test_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

split_task_gemma_result = pd.DataFrame([
    {
        "model": "Split-task: Gemma LoRA entities + SciBERT relation classifier",
        "split": "test",
        **split_task_gemma_metrics
    }
])

split_task_gemma_result.to_csv(
    f"{output_dir}/split_task_gemma_entities_scibert_relations_test_metrics.csv",
    index=False
)

threshold_results_df.to_csv(
    f"{output_dir}/split_task_gemma_scibert_threshold_results.csv",
    index=False
)

split_task_gemma_result

#OOD Evaluation — Best Method

In [ ]:
ood_entity_temp_path = f"{output_dir}/gemma_lora_entity_only_ood_temp_predictions.jsonl"
ood_entity_final_path = f"{output_dir}/gemma_lora_entity_only_ood_predictions.jsonl"

gemma_entity_ood_df = test_ood_df.copy().reset_index(drop=True)

if os.path.exists(ood_entity_temp_path):
    temp_df = pd.read_json(ood_entity_temp_path, orient="records", lines=True)
    start_idx = len(temp_df)

    raw_predictions = temp_df["entity_prediction_raw"].tolist()
    repaired_predictions = temp_df["entity_prediction_text"].tolist()

    print("Resuming OOD entity prediction from:", start_idx)
else:
    start_idx = 0
    raw_predictions = []
    repaired_predictions = []

    print("Starting OOD entity prediction from beginning")

for idx in tqdm(range(start_idx, len(gemma_entity_ood_df))):
    sentence = gemma_entity_ood_df.iloc[idx]["sentence"]

    raw_output = generate_gemma_entities(sentence, max_new_tokens=96)
    repaired_output = repair_entity_prediction(raw_output, sentence=sentence)

    raw_predictions.append(raw_output)
    repaired_predictions.append(repaired_output)

    if (idx + 1) % 20 == 0:
        temp_df = gemma_entity_ood_df.iloc[:idx + 1].copy()
        temp_df["entity_prediction_raw"] = raw_predictions
        temp_df["entity_prediction_text"] = repaired_predictions

        temp_df.to_json(
            ood_entity_temp_path,
            orient="records",
            lines=True,
            force_ascii=False
        )

        print(f"Saved OOD entity progress at {idx + 1}/{len(gemma_entity_ood_df)}")

gemma_entity_ood_df["entity_prediction_raw"] = raw_predictions
gemma_entity_ood_df["entity_prediction_text"] = repaired_predictions

gemma_entity_ood_df.to_json(
    ood_entity_final_path,
    orient="records",
    lines=True,
    force_ascii=False
)

print("Saved OOD entity-only predictions:", ood_entity_final_path)

In [ ]:
entity_only_ood_eval_df = gemma_entity_ood_df.copy()

full_json_predictions = []

for _, row in entity_only_ood_eval_df.iterrows():
    obj, _ = parse_json_safe(row["entity_prediction_text"])

    final_obj = {
        "entities": obj.get("entities", []),
        "relations": []
    }

    full_json_predictions.append(
        json.dumps(final_obj, ensure_ascii=False, sort_keys=True)
    )

entity_only_ood_eval_df["prediction_text"] = full_json_predictions

entity_only_ood_metrics = evaluate_structured_predictions(entity_only_ood_eval_df)
entity_only_ood_metrics = add_equal_weighted_final_scores(entity_only_ood_metrics)
entity_only_ood_metrics = add_relation_weighted_final_scores(entity_only_ood_metrics)
entity_only_ood_metrics = add_partial_entity_recall(
    entity_only_ood_metrics,
    entity_only_ood_eval_df
)

entity_only_ood_metrics

In [ ]:
# Use the best threshold found on the normal test set
# If best_threshold is not defined, use 0.50
if "best_threshold" not in globals():
    best_threshold = 0.50

print("Using threshold:", best_threshold)

split_task_gemma_ood_df = gemma_entity_ood_df.copy()
ood_split_predictions = []

for _, row in tqdm(split_task_gemma_ood_df.iterrows(), total=len(split_task_gemma_ood_df)):
    pred_text = build_split_task_json_from_gemma(
        row,
        threshold=best_threshold
    )

    ood_split_predictions.append(pred_text)

split_task_gemma_ood_df["prediction_text"] = ood_split_predictions

split_task_gemma_ood_df.head()

In [ ]:
split_task_gemma_ood_metrics = evaluate_structured_predictions(split_task_gemma_ood_df)

split_task_gemma_ood_metrics = add_equal_weighted_final_scores(
    split_task_gemma_ood_metrics
)

split_task_gemma_ood_metrics = add_relation_weighted_final_scores(
    split_task_gemma_ood_metrics
)

split_task_gemma_ood_metrics = add_partial_entity_recall(
    split_task_gemma_ood_metrics,
    split_task_gemma_ood_df
)

split_task_gemma_ood_metrics

In [ ]:
split_task_gemma_ood_df.to_json(
    f"{output_dir}/split_task_gemma_entities_scibert_relations_test_ood_predictions.jsonl",
    orient="records",
    lines=True,
    force_ascii=False
)

split_task_gemma_ood_result = pd.DataFrame([
    {
        "model": "Split-task: Gemma LoRA entities + SciBERT relation classifier",
        "split": "test_ood",
        **split_task_gemma_ood_metrics
    }
])

split_task_gemma_ood_result.to_csv(
    f"{output_dir}/split_task_gemma_entities_scibert_relations_test_ood_metrics.csv",
    index=False
)

split_task_gemma_ood_result

In [ ]:
test_vs_ood = pd.DataFrame([
    {
        "split": "test",
        "valid_json_rate": split_task_gemma_metrics["valid_json_rate"],
        "exact_match_rate": split_task_gemma_metrics["exact_match_rate"],
        "entity_micro_f1": split_task_gemma_metrics["entity_micro_f1"],
        "relation_micro_f1": split_task_gemma_metrics["relation_micro_f1"],
        "final_micro_f1_equal_weighted": split_task_gemma_metrics["final_micro_f1_equal_weighted"],
        "partial_entity_recall": split_task_gemma_metrics["partial_entity_recall"]
    },
    {
        "split": "test_ood",
        "valid_json_rate": split_task_gemma_ood_metrics["valid_json_rate"],
        "exact_match_rate": split_task_gemma_ood_metrics["exact_match_rate"],
        "entity_micro_f1": split_task_gemma_ood_metrics["entity_micro_f1"],
        "relation_micro_f1": split_task_gemma_ood_metrics["relation_micro_f1"],
        "final_micro_f1_equal_weighted": split_task_gemma_ood_metrics["final_micro_f1_equal_weighted"],
        "partial_entity_recall": split_task_gemma_ood_metrics["partial_entity_recall"]
    }
])

test_vs_ood.round(4)

In [ ]:
split_task_gemma_ood_metrics = {
    "valid_json_rate": 1.0,
    "exact_match_rate": 0.296551724137931,
    "entity_micro_precision": 0.6768488745980707,
    "entity_micro_recall": 0.6719872306464485,
    "entity_micro_f1": 0.6744092911493793,
    "entity_example_precision": 0.6637643678160919,
    "entity_example_recall": 0.6888793629741906,
    "entity_example_f1": 0.6514464069908716,
    "relation_micro_precision": 0.36983842010771995,
    "relation_micro_recall": 0.356401384083045,
    "relation_micro_f1": 0.36299559471365644,
    "relation_example_precision": 0.5960851122058018,
    "relation_example_recall": 0.5900807334428024,
    "relation_example_f1": 0.5841437011264597,
    "final_micro_f1_equal_weighted": 0.5187024429315179,
    "final_example_f1_equal_weighted": 0.6177950540586656,
    "final_micro_f1_relation_weighted": 0.4875610732879456,
    "final_example_f1_relation_weighted": 0.6110647834722245,
    "partial_entity_recall": 0.7858149130562924
}

split_task_gemma_ood_metrics

In [ ]:
import os


project_dir = "/content/drive/MyDrive/NLP_26_Project"
output_dir = f"{project_dir}/SciER_outputs"

os.makedirs(output_dir, exist_ok=True)

print("output_dir:", output_dir)

In [ ]:
import os
import pandas as pd
import numpy as np


project_dir = "/content/drive/MyDrive/NLP_26_Project"
output_dir = f"{project_dir}/SciER_outputs"
os.makedirs(output_dir, exist_ok=True)

comparison_path = f"{output_dir}/comparison_results.csv"
clean_comparison_path = f"{output_dir}/comparison_results_clean.csv"


def load_existing_comparison_table(comparison_path):
    if os.path.exists(comparison_path):
        print("Loading existing comparison table:", comparison_path)
        return pd.read_csv(comparison_path)

    print("No existing comparison table found. Starting empty.")
    return pd.DataFrame()


def upsert_metric_row(df, model_name, split_name, metrics):
    new_row = pd.DataFrame([
        {
            "model": model_name,
            "split": split_name,
            **metrics
        }
    ])

    if df.empty:
        return new_row

    df = df[
        ~(
            (df["model"] == model_name) &
            (df["split"] == split_name)
        )
    ].reset_index(drop=True)

    df = pd.concat([df, new_row], ignore_index=True)

    return df


def rebuild_comparison_from_existing_and_metrics(
    comparison_path,
    output_path
):
    comparison_results = load_existing_comparison_table(comparison_path)

    metric_sources = [
        ("Zero-shot Qwen2.5-1.5B-Instruct: Direct JSON Generation", "test", "qwen_zero_metrics"),
        ("Few-shot Qwen2.5-1.5B-Instruct: Direct JSON Generation", "test", "qwen_few_metrics"),
        ("LoRA fine-tuned Qwen2.5-1.5B-Instruct train3000", "test", "qwen_lora_full_metrics"),

        ("Zero-shot Gemma-2-2B-it", "test", "gemma_zero_metrics"),
        ("Few-shot Gemma-2-2B-it", "test", "gemma_few_metrics"),
        ("LoRA fine-tuned Gemma-2-2B-it train5000", "test", "gemma_lora_metrics"),

        ("Zero-shot Llama-3.2-1B-Instruct", "test", "llama_zero_metrics"),
        ("Few-shot Llama-3.2-1B-Instruct", "test", "llama_few_metrics"),
        ("LoRA fine-tuned Llama-3.2-1B-Instruct train3000", "test", "llama_lora_metrics"),

        ("Entity-only Gemma LoRA", "test", "entity_only_metrics"),
        ("Split-task: Gemma LoRA entities + SciBERT relation classifier", "test", "split_task_gemma_metrics"),
        ("Split-task: Gemma LoRA entities + SciBERT relation classifier", "test_ood", "split_task_gemma_ood_metrics"),

        ("Upper-bound: Gold entities + SciBERT relation classifier with entity markers", "test", "gold_scibert_metrics"),
    ]

    missing = []

    for model_name, split_name, var_name in metric_sources:
        if var_name in globals():
            comparison_results = upsert_metric_row(
                comparison_results,
                model_name,
                split_name,
                globals()[var_name]
            )
        else:
            missing.append(var_name)

    if missing:
        print("These metric dictionaries were not found in memory:")
        for name in missing:
            print("-", name)


    if {"entity_micro_f1", "relation_micro_f1"}.issubset(comparison_results.columns):
        mask_micro = (
            comparison_results["entity_micro_f1"].notna() &
            comparison_results["relation_micro_f1"].notna()
        )

        comparison_results.loc[mask_micro, "final_micro_f1_equal_weighted"] = (
            0.5 * comparison_results.loc[mask_micro, "entity_micro_f1"] +
            0.5 * comparison_results.loc[mask_micro, "relation_micro_f1"]
        )

        comparison_results.loc[mask_micro, "final_micro_f1_relation_weighted"] = (
            0.4 * comparison_results.loc[mask_micro, "entity_micro_f1"] +
            0.6 * comparison_results.loc[mask_micro, "relation_micro_f1"]
        )

    if {"entity_example_f1", "relation_example_f1"}.issubset(comparison_results.columns):
        mask_example = (
            comparison_results["entity_example_f1"].notna() &
            comparison_results["relation_example_f1"].notna()
        )

        comparison_results.loc[mask_example, "final_example_f1_equal_weighted"] = (
            0.5 * comparison_results.loc[mask_example, "entity_example_f1"] +
            0.5 * comparison_results.loc[mask_example, "relation_example_f1"]
        )

        comparison_results.loc[mask_example, "final_example_f1_relation_weighted"] = (
            0.4 * comparison_results.loc[mask_example, "entity_example_f1"] +
            0.6 * comparison_results.loc[mask_example, "relation_example_f1"]
        )

    preferred_cols = [
        "model",
        "split",
        "valid_json_rate",
        "exact_match_rate",
        "entity_micro_precision",
        "entity_micro_recall",
        "entity_micro_f1",
        "entity_example_precision",
        "entity_example_recall",
        "entity_example_f1",
        "relation_micro_precision",
        "relation_micro_recall",
        "relation_micro_f1",
        "relation_example_precision",
        "relation_example_recall",
        "relation_example_f1",
        "final_micro_f1_equal_weighted",
        "final_example_f1_equal_weighted",
        "final_micro_f1_relation_weighted",
        "final_example_f1_relation_weighted",
        "partial_entity_recall"
    ]

    existing_preferred_cols = [
        col for col in preferred_cols
        if col in comparison_results.columns
    ]

    extra_cols = [
        col for col in comparison_results.columns
        if col not in existing_preferred_cols
    ]

    comparison_results = comparison_results[
        existing_preferred_cols + extra_cols
    ]

    comparison_results = comparison_results.drop_duplicates(
        subset=["model", "split"],
        keep="last"
    ).reset_index(drop=True)

    comparison_results.to_csv(output_path, index=False)

    print("Saved clean comparison table:", output_path)
    print("Number of rows:", len(comparison_results))

    return comparison_results


comparison_results = rebuild_comparison_from_existing_and_metrics(
    comparison_path=comparison_path,
    output_path=clean_comparison_path
)

comparison_results

In [ ]:

comparison_results = pd.DataFrame([
    {
        "model": "Empty baseline",
        "split": "test",
        "valid_json_rate": 1.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.11416861826697892,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.1370023419203747,
        "partial_entity_recall": 0.0
    },
    {
        "model": "Rule-based baseline",
        "split": "test",
        "valid_json_rate": 1.0,
        "exact_match_rate": 0.00468384074941452,
        "entity_micro_precision": 0.2233219919579338,
        "entity_micro_recall": 0.26102675343456255,
        "entity_micro_f1": 0.24070678446407734,
        "entity_example_precision": 0.2622604334682943,
        "entity_example_recall": 0.26403423209629295,
        "entity_example_f1": 0.24475568573757467,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.12035339223203867,
        "final_example_f1_equal_weighted": 0.23654646113576627,
        "final_micro_f1_relation_weighted": 0.09628271378563094,
        "final_example_f1_relation_weighted": 0.23490461621540457,
        "partial_entity_recall": np.nan
    },
    {
        "model": "Rule-based baseline",
        "split": "test_ood",
        "valid_json_rate": 1.0,
        "exact_match_rate": 0.08448275862068966,
        "entity_micro_precision": 0.37845923709798057,
        "entity_micro_recall": 0.4038308060654429,
        "entity_micro_f1": 0.39073359073359076,
        "entity_example_precision": 0.38785589889038163,
        "entity_example_recall": 0.4377832512315271,
        "entity_example_f1": 0.37730686182138085,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.5431034482758621,
        "relation_example_recall": 0.5431034482758621,
        "relation_example_f1": 0.5431034482758621,
        "final_micro_f1_equal_weighted": 0.19536679536679538,
        "final_example_f1_equal_weighted": 0.4602051550486215,
        "final_micro_f1_relation_weighted": np.nan,
        "final_example_f1_relation_weighted": np.nan,
        "partial_entity_recall": np.nan
    },
    {
        "model": "Zero-shot FLAN-T5-small: Direct JSON Generation",
        "split": "test_20",
        "valid_json_rate": 0.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.2,
        "relation_example_recall": 0.2,
        "relation_example_f1": 0.2,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.1,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.12,
        "partial_entity_recall": np.nan
    },
    {
        "model": "Strong JSON Prompt FLAN-T5-small",
        "split": "test",
        "valid_json_rate": 0.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.11416861826697892,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.1370023419203747,
        "partial_entity_recall": 0.0
    },
    {
        "model": "10-shot FLAN-T5-small + Line/Flexible Parser",
        "split": "test",
        "valid_json_rate": 1.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.11416861826697892,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.1370023419203747,
        "partial_entity_recall": 0.0
    },
    {
        "model": "Zero-shot FLAN-T5-base + Label-Constrained Prompt + Flexible Parser",
        "split": "test",
        "valid_json_rate": 1.0,
        "exact_match_rate": 0.00117096018735363,
        "entity_micro_precision": 0.10682288077188146,
        "entity_micro_recall": 0.05603759942154736,
        "entity_micro_f1": 0.07351197533791795,
        "entity_example_precision": 0.07074165101097185,
        "entity_example_recall": 0.05924872681312963,
        "entity_example_f1": 0.05956201823020509,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.036755987668958975,
        "final_example_f1_equal_weighted": 0.14394962738208147,
        "final_micro_f1_relation_weighted": 0.029404790135167183,
        "final_example_f1_relation_weighted": 0.16082714921245672,
        "partial_entity_recall": 0.11159343518828296
    },
    {
        "model": "Few-shot FLAN-T5-base: Direct JSON Generation",
        "split": "test",
        "valid_json_rate": 0.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22833723653395785,
        "relation_example_recall": 0.22833723653395785,
        "relation_example_f1": 0.22833723653395785,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.11416861826697892,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.1370023419203747,
        "partial_entity_recall": 0.0
    },
    {
        "model": "Partial fine-tuned FLAN-T5-base: last decoder block + lm_head",
        "split": "test_200",
        "valid_json_rate": 0.0,
        "exact_match_rate": 0.0,
        "entity_micro_precision": 0.0,
        "entity_micro_recall": 0.0,
        "entity_micro_f1": 0.0,
        "entity_example_precision": 0.0,
        "entity_example_recall": 0.0,
        "entity_example_f1": 0.0,
        "relation_micro_precision": 0.0,
        "relation_micro_recall": 0.0,
        "relation_micro_f1": 0.0,
        "relation_example_precision": 0.22,
        "relation_example_recall": 0.22,
        "relation_example_f1": 0.22,
        "final_micro_f1_equal_weighted": 0.0,
        "final_example_f1_equal_weighted": 0.11,
        "final_micro_f1_relation_weighted": 0.0,
        "final_example_f1_relation_weighted": 0.132,
        "partial_entity_recall": 0.0
    }
])

comparison_results

In [ ]:
comparison_results.to_csv(
    f"{output_dir}/comparison_results.csv",
    index=False
)

print("Saved updated comparison table to:", f"{output_dir}/comparison_results.csv")

In [ ]:
import pandas as pd

comparison_results = pd.read_csv(
    "/content/drive/MyDrive/NLP_26_Project/SciER_outputs/comparison_results.csv"
)

comparison_results